# SIPTA: Notebook 04 — Modelado y Cálculo del Índice de Prioridad Territorial (IPT)

**Fase PDCO**: DEVELOPMENT | **Etapa Workflow**: 1.6 Modelado e Indicadores  

Este notebook calcula indicadores normalizados y construye el Índice de Prioridad Territorial (IPT) a partir de `src.modeling.calculate_indicators`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.modeling.calculate_indicators as mdl

print(f"Raíz: {ROOT}")
print(f"Directorio processed: {mdl.PROCESSED_DIR}")
print(f"Directorio curated: {mdl.CURATED_DIR}")



Raíz: C:\Users\ADAN\DataJam_DataOlinguitos_Gen
Directorio processed: C:\Users\ADAN\DataJam_DataOlinguitos_Gen\data\processed
Directorio curated: C:\Users\ADAN\DataJam_DataOlinguitos_Gen\data\curated


## 2. Control de fuentes habilitadas para el modelado

El IPT utilizará exclusivamente fuentes registradas en `data/status`.
Las fuentes con estado `approved_partial` solo podrán emplearse con una
limitación metodológica explícita. El catálogo será la base de trazabilidad
entre cada indicador, su origen y su temporalidad.

In [2]:
STATUS_DIR = ROOT / "data" / "status"

approved_sources = pd.read_csv(STATUS_DIR / "approved_sources.csv")
source_catalog = pd.read_csv(STATUS_DIR / "source_catalog.csv")

fuentes_modelo = (
    approved_sources.loc[
        approved_sources["estado"].isin(["approved", "approved_partial"]),
        [
            "id",
            "nombre",
            "archivo",
            "indicadores",
            "estado",
            "existe",
            "lectura",
            "conteo",
        ],
    ]
    .merge(
        source_catalog[
            ["id", "origen", "temporalidad", "valor_publico"]
        ],
        on="id",
        how="left",
    )
    .sort_values(["estado", "id"])
    .reset_index(drop=True)
)

print("Fuentes habilitadas:", len(fuentes_modelo))
print(fuentes_modelo["estado"].value_counts())
display(fuentes_modelo)

Fuentes habilitadas: 31
estado
approved            29
approved_partial     2
Name: count, dtype: int64


,id,nombre,archivo,indicadores,estado,existe,lectura,conteo,origen,temporalidad,valor_publico
0,AMB-AIRE,Estaciones Red Calidad del Aire (SDA),Ambiente/estacion_calidad_aire.geojson,"AMB-01, AMB-03",approved,True,True,True,ambientebogota.gov.co (RMCAB / SDA),Red Activa 2026,Monitoreo continuo de contaminacion atmosferic...
1,AMB-SAC,Situacion ambiental conflictiva (SDA),Ambiente/situacion_ambiental_conflictiva.csv,"AMB-01, AMB-02",approved,True,True,True,ambientebogota.gov.co (SDA / IDECA),2020-2025,Conflictos ambientales y riesgos territoriales
2,EDU-COLEGIOS,Sedes educativas de Bogota (SED),Educacion/colegios122025.gpkg,"EDU-01, EDU-02",approved,True,True,True,datosabiertos.bogota.gov.co/dataset/colegios-b...,Corte 12.2025,Acceso a educacion y su conectividad con el tr...
3,EDU-CUPOS,Oferta de cupos sector oficial (SED - Yesid),Educacion/ofertacupos_032025.geojson,"EDU-01, EDU-02",approved,True,True,True,educacionbogota.edu.co (SED),Corte 03.2025,Capacidad de oferta educativa oficial por loca...
4,EDU-MATRICULA,Matricula total colegios oficiales (SED),Educacion/matricula_total_colegios_oficiales.gpkg,"EDU-01, EDU-02",approved,True,True,True,datosabiertos.bogota.gov.co (SED),Corte 04.2025,Capacidad escolar oficial por localidad
5,FIN-PUNTOS,Puntos de encuentro vendedores (IPES),Finanzas/Punto de encuentro vendedores. Bogotá...,FIN-02,approved,True,True,True,ipes.gov.co (IPES),2024,Infraestructura distrital para formalizacion c...
6,FIN-RIVI,Vendedores informales RIVI 2017-2019 (IPES),Finanzas/rivi-numero-vendedores-informales-loc...,"FIN-01, FIN-02",approved,True,True,True,ipes.gov.co (IPES RIVI),Semestral 2017-2019,Vulnerabilidad del comercio informal e ingresos
7,INFRA-PARQUES,Inventario distrital de parques (IDRD),Infraestructura/5.-parques-idrd.csv,"INF-01, INF-04",approved,True,True,True,datosabiertos.bogota.gov.co (IDRD),Corte 2024-2025,Espacio publico recreativo y verde por habitante
8,IPS-LOCAL,IPS (copia local del catalogo DataJam),Infraestructura/ips.gpkg,"SAL-01, SAL-02",approved,True,True,True,Catalogo DataJam_DataOlinguitos_Gen (referencia),Vigente,Cruz con SAL-IPS (verificacion de paridad)
9,METRO-L1,Estaciones Linea 1 del Metro (proyectadas),Metro/estaciones_linea1.geojson,MOV-15,approved,True,True,True,datosabiertos.bogota.gov.co (Metro de Bogota),Escenario futuro (proyecto),Cobertura futura del transporte masivo (radio ...


## 3. Contrato de fuentes del IPT base

El modelo base tendrá como unidad de análisis las 20 localidades de Bogotá.
Se utilizarán fuentes con estado `approved`, comparables territorialmente y
asociadas con los indicadores definidos en la matriz de trazabilidad.

Las fuentes parciales y los escenarios futuros se analizarán por separado y
no participarán inicialmente en el puntaje de necesidad territorial.

In [3]:
seleccion_ipt = pd.DataFrame(
    [
        {
            "id": "MR",
            "dimension": "Base territorial",
            "rol_modelo": "Geometría y área oficial",
            "indicador_objetivo": "DIM_TERRITORIO",
        },
        {
            "id": "POB-LOC",
            "dimension": "Demografía",
            "rol_modelo": "Denominador poblacional",
            "indicador_objetivo": "Población por localidad",
        },
        {
            "id": "SAL-IPS",
            "dimension": "Salud",
            "rol_modelo": "Oferta sanitaria",
            "indicador_objetivo": "IPS por 10.000 habitantes",
        },
        {
            "id": "EDU-CUPOS",
            "dimension": "Educación",
            "rol_modelo": "Oferta educativa",
            "indicador_objetivo": "Cupos por población objetivo",
        },
        {
            "id": "TM-ESTACIONES",
            "dimension": "Movilidad",
            "rol_modelo": "Acceso troncal",
            "indicador_objetivo": "Estaciones por km²",
        },
        {
            "id": "TM-PARADEROS",
            "dimension": "Movilidad",
            "rol_modelo": "Acceso zonal",
            "indicador_objetivo": "Paraderos por km²",
        },
        {
            "id": "INFRA-PARQUES",
            "dimension": "Infraestructura",
            "rol_modelo": "Espacio público",
            "indicador_objetivo": "Área de parques por habitante",
        },
        {
            "id": "AMB-SAC",
            "dimension": "Ambiente",
            "rol_modelo": "Conflictividad ambiental",
            "indicador_objetivo": "Conflictos ambientales por km²",
        },
        {
            "id": "FIN-RIVI",
            "dimension": "Vulnerabilidad económica",
            "rol_modelo": "Comercio informal",
            "indicador_objetivo": "Vendedores informales por 10.000 habitantes",
        },
        {
            "id": "SEG-CUADRANTES",
            "dimension": "Seguridad",
            "rol_modelo": "Cobertura preventiva",
            "indicador_objetivo": "Cuadrantes por 10.000 habitantes",
        },
    ]
)

fuentes_ipt = seleccion_ipt.merge(
    fuentes_modelo[
        [
            "id",
            "nombre",
            "archivo",
            "estado",
            "origen",
            "temporalidad",
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

assert fuentes_ipt["estado"].notna().all(), "Hay fuentes que no aparecen en data/status"
assert fuentes_ipt["estado"].eq("approved").all(), "El IPT base contiene fuentes no aprobadas"

print("Fuentes seleccionadas para el IPT base:", len(fuentes_ipt))
print("Estados:")
print(fuentes_ipt["estado"].value_counts())

display(
    fuentes_ipt[
        [
            "dimension",
            "id",
            "indicador_objetivo",
            "estado",
            "temporalidad",
        ]
    ]
)

Fuentes seleccionadas para el IPT base: 10
Estados:
estado
approved    10
Name: count, dtype: int64


,dimension,id,indicador_objetivo,estado,temporalidad
0,Base territorial,MR,DIM_TERRITORIO,approved,Version v03.26 (2025)
1,Demografía,POB-LOC,Población por localidad,approved,2005-2035 (proyeccion anual)
2,Salud,SAL-IPS,IPS por 10.000 habitantes,approved,Vigente (2025)
3,Educación,EDU-CUPOS,Cupos por población objetivo,approved,Corte 03.2025
4,Movilidad,TM-ESTACIONES,Estaciones por km²,approved,Vigente (2025-2026)
5,Movilidad,TM-PARADEROS,Paraderos por km²,approved,Vigente (2025-2026)
6,Infraestructura,INFRA-PARQUES,Área de parques por habitante,approved,Corte 2024-2025
7,Ambiente,AMB-SAC,Conflictos ambientales por km²,approved,2020-2025
8,Vulnerabilidad económica,FIN-RIVI,Vendedores informales por 10.000 habitantes,approved,Semestral 2017-2019
9,Seguridad,SEG-CUADRANTES,Cuadrantes por 10.000 habitantes,approved,Vigente


## 4. Disponibilidad física de las fuentes seleccionadas

Se verificará la ubicación de cada archivo antes de leerlo. El modelado
priorizará archivos de `data/processed`; los archivos encontrados únicamente
en `data/raw` quedarán identificados para no mezclar etapas del pipeline.

In [4]:
patrones_archivos = {
    "MR": "poligonos_localidades.geojson",
    "POB-LOC": "osb_demografia-poblacion-localidad.csv",
    "SAL-IPS": "ips_sds.gpkg",
    "EDU-CUPOS": "ofertacupos_032025.geojson",
    "TM-ESTACIONES": "estaciones_troncales.geojson",
    "TM-PARADEROS": "paraderos_zonales_sitp.gpkg",
    "INFRA-PARQUES": "5.-parques-idrd.csv",
    "AMB-SAC": "situacion_ambiental_conflictiva.csv",
    "FIN-RIVI": "rivi-numero-vendedores-informales-localidad-*",
    "SEG-CUADRANTES": "Cuadrante de Policía. Bogotá D.C.csv",
}

registros_ubicacion = []

for source_id, patron in patrones_archivos.items():
    coincidencias = sorted(
        ruta for ruta in (ROOT / "data").rglob(patron)
        if ruta.is_file()
    )

    if not coincidencias:
        registros_ubicacion.append(
            {
                "id": source_id,
                "encontrado": False,
                "etapa": None,
                "ruta": None,
            }
        )
        continue

    for ruta in coincidencias:
        etapa = next(
            (
                nombre
                for nombre in ("processed", "curated", "raw")
                if nombre in ruta.parts
            ),
            "otra",
        )

        registros_ubicacion.append(
            {
                "id": source_id,
                "encontrado": True,
                "etapa": etapa,
                "ruta": ruta.relative_to(ROOT).as_posix(),
            }
        )

ubicaciones_fuentes = pd.DataFrame(registros_ubicacion)

resumen_ubicacion = (
    ubicaciones_fuentes.groupby("id", as_index=False)
    .agg(
        encontrado=("encontrado", "max"),
        cantidad_archivos=("ruta", "count"),
        etapas=("etapa", lambda valores: ", ".join(sorted(set(valores.dropna())))),
    )
    .merge(
        seleccion_ipt[["id", "dimension"]],
        on="id",
        how="left",
    )
    .sort_values("dimension")
)

print(
    "Fuentes localizadas:",
    int(resumen_ubicacion["encontrado"].sum()),
    "de",
    len(resumen_ubicacion),
)

display(resumen_ubicacion)
display(ubicaciones_fuentes.sort_values(["id", "etapa"]))

Fuentes localizadas: 10 de 10


,id,encontrado,cantidad_archivos,etapas,dimension
0,AMB-SAC,True,2,"processed, raw",Ambiente
4,MR,True,2,"processed, raw",Base territorial
5,POB-LOC,True,4,"processed, raw",Demografía
1,EDU-CUPOS,True,2,"processed, raw",Educación
3,INFRA-PARQUES,True,2,"processed, raw",Infraestructura
8,TM-ESTACIONES,True,2,"processed, raw",Movilidad
9,TM-PARADEROS,True,2,"processed, raw",Movilidad
6,SAL-IPS,True,2,"processed, raw",Salud
7,SEG-CUADRANTES,True,2,"processed, raw",Seguridad
2,FIN-RIVI,True,6,raw,Vulnerabilidad económica


,id,encontrado,etapa,ruta
16,AMB-SAC,True,processed,data/processed/AMBIENTE/situacion_ambiental_co...
17,AMB-SAC,True,raw,data/raw/AMBIENTE/situacion_ambiental_conflict...
8,EDU-CUPOS,True,processed,data/processed/EDUCACION/ofertacupos_032025.ge...
9,EDU-CUPOS,True,raw,data/raw/EDUCACION/ofertacupos_032025.geojson
18,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...
19,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...
20,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...
21,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...
22,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...
23,FIN-RIVI,True,raw,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...


### 4.1 Rutas canónicas de entrada

Para evitar duplicidades, cada fuente utilizará una ruta canónica en
`data/processed`. La excepción es `FIN-RIVI`, cuyos seis archivos históricos
aprobados se encuentran únicamente en `data/raw`.

Los archivos raw de RIVI serán leídos sin modificarlos y su temporalidad
2017–2019 se conservará explícitamente como limitación.

In [5]:
rutas_unicas = {
    "MR": ROOT / "data/processed/MODELO_TERRITORIAL/poligonos_localidades.geojson",
    "POB-LOC": ROOT / "data/processed/DEMOGRAFIA/osb_demografia-poblacion-localidad.csv",
    "SAL-IPS": ROOT / "data/processed/SALUD/ips_sds.gpkg",
    "EDU-CUPOS": ROOT / "data/processed/EDUCACION/ofertacupos_032025.geojson",
    "TM-ESTACIONES": ROOT / "data/processed/MOVILIDAD/estaciones_troncales.geojson",
    "TM-PARADEROS": ROOT / "data/processed/MOVILIDAD/paraderos_zonales_sitp.gpkg",
    "INFRA-PARQUES": ROOT / "data/processed/INFRAESTRUCTURA_ESPACIO_PUBLICO/5.-parques-idrd.csv",
    "AMB-SAC": ROOT / "data/processed/AMBIENTE/situacion_ambiental_conflictiva.csv",
    "SEG-CUADRANTES": ROOT / "data/processed/SEGURIDAD/Cuadrante de Policía. Bogotá D.C.csv",
}

rutas_rivi = sorted(
    (
        ROOT
        / "data/raw/FINANZAS_INVERSION_PUBLICA"
    ).glob("rivi-numero-vendedores-informales-localidad-*")
)

assert all(ruta.exists() for ruta in rutas_unicas.values())
assert len(rutas_rivi) == 6, (
    f"Se esperaban 6 archivos RIVI y se encontraron {len(rutas_rivi)}"
)

registros_entrada = [
    {
        "id": source_id,
        "etapa": "processed",
        "archivo": ruta.name,
        "ruta": ruta.relative_to(ROOT).as_posix(),
        "existe": ruta.exists(),
    }
    for source_id, ruta in rutas_unicas.items()
]

registros_entrada.extend(
    {
        "id": "FIN-RIVI",
        "etapa": "raw",
        "archivo": ruta.name,
        "ruta": ruta.relative_to(ROOT).as_posix(),
        "existe": ruta.exists(),
    }
    for ruta in rutas_rivi
)

entradas_ipt = pd.DataFrame(registros_entrada)

print("Archivos de entrada:", len(entradas_ipt))
print("Fuentes representadas:", entradas_ipt["id"].nunique())
print("Todos existen:", entradas_ipt["existe"].all())

display(entradas_ipt.sort_values(["id", "archivo"]))

Archivos de entrada: 15
Fuentes representadas: 10
Todos existen: True


,id,etapa,archivo,ruta,existe
7,AMB-SAC,processed,situacion_ambiental_conflictiva.csv,data/processed/AMBIENTE/situacion_ambiental_co...,True
3,EDU-CUPOS,processed,ofertacupos_032025.geojson,data/processed/EDUCACION/ofertacupos_032025.ge...,True
9,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
10,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
11,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
12,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
13,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
14,FIN-RIVI,raw,rivi-numero-vendedores-informales-localidad-20...,data/raw/FINANZAS_INVERSION_PUBLICA/rivi-numer...,True
6,INFRA-PARQUES,processed,5.-parques-idrd.csv,data/processed/INFRAESTRUCTURA_ESPACIO_PUBLICO...,True
0,MR,processed,poligonos_localidades.geojson,data/processed/MODELO_TERRITORIAL/poligonos_lo...,True


## 5. Auditoría de esquemas de entrada

Se inspeccionarán la cantidad de registros, columnas territoriales, variables
numéricas y sistemas de referencia espacial. Esta revisión permite seleccionar
el numerador y denominador correctos antes de construir indicadores.

In [6]:
import re
import geopandas as gpd


def leer_archivo_modelo(ruta):
    extension = ruta.suffix.lower()

    if extension in {".gpkg", ".geojson", ".shp"}:
        return gpd.read_file(ruta)

    if extension in {".csv", ".txt"}:
        errores = []

        for encoding in ("utf-8-sig", "latin-1"):
            try:
                return pd.read_csv(
                    ruta,
                    encoding=encoding,
                    sep=None,
                    engine="python",
                )
            except Exception as exc:
                errores.append(
                    f"{encoding}: {type(exc).__name__}: {exc}"
                )

        detalle = " | ".join(errores)
        raise ValueError(
            f"No se pudo leer {ruta.name}. Intentos: {detalle}"
        )

    raise ValueError(
        f"Formato no soportado: {ruta.name} ({extension})"
    )


rutas_por_fuente = {
    source_id: [ruta]
    for source_id, ruta in rutas_unicas.items()
}
rutas_por_fuente["FIN-RIVI"] = rutas_rivi

datos_fuentes = {}
registros_esquema = []

for source_id, rutas in rutas_por_fuente.items():
    datos_fuentes[source_id] = []

    for ruta in rutas:
        try:
            df = leer_archivo_modelo(ruta)
            datos_fuentes[source_id].append(
                {
                    "ruta": ruta,
                    "datos": df,
                }
            )

            columnas_territoriales = [
                str(columna)
                for columna in df.columns
                if re.search(
                    r"localidad|cod.*loc|nom.*loc|locali",
                    str(columna),
                    flags=re.IGNORECASE,
                )
            ]

            columnas_numericas = [
                str(columna)
                for columna in df.select_dtypes(include="number").columns
            ]

            registros_esquema.append(
                {
                    "id": source_id,
                    "archivo": ruta.name,
                    "filas": len(df),
                    "n_columnas": len(df.columns),
                    "crs": str(getattr(df, "crs", None)),
                    "columnas_territoriales": " | ".join(columnas_territoriales),
                    "columnas_numericas": " | ".join(columnas_numericas),
                    "error": None,
                }
            )

        except Exception as exc:
            registros_esquema.append(
                {
                    "id": source_id,
                    "archivo": ruta.name,
                    "filas": None,
                    "n_columnas": None,
                    "crs": None,
                    "columnas_territoriales": None,
                    "columnas_numericas": None,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

esquemas_archivos = pd.DataFrame(registros_esquema)


def unir_columnas(valores):
    resultado = set()

    for texto in valores.dropna():
        resultado.update(
            valor.strip()
            for valor in texto.split("|")
            if valor.strip()
        )

    return " | ".join(sorted(resultado))


esquemas_por_fuente = (
    esquemas_archivos.groupby("id", as_index=False)
    .agg(
        archivos=("archivo", "count"),
        filas_totales=("filas", "sum"),
        columnas_territoriales=("columnas_territoriales", unir_columnas),
        columnas_numericas=("columnas_numericas", unir_columnas),
        errores=("error", lambda valores: valores.notna().sum()),
    )
    .sort_values("id")
)

print("Archivos leídos:", esquemas_archivos["error"].isna().sum())
print("Archivos con error:", esquemas_archivos["error"].notna().sum())
print("Fuentes auditadas:", len(esquemas_por_fuente))

display(
    esquemas_por_fuente[
        [
            "id",
            "archivos",
            "filas_totales",
            "errores",
            "columnas_territoriales",
            "columnas_numericas",
        ]
    ]
)

if esquemas_archivos["error"].notna().any():
    display(
        esquemas_archivos.loc[
            esquemas_archivos["error"].notna(),
            ["id", "archivo", "error"],
        ]
    )

Archivos leídos: 15
Archivos con error: 0
Fuentes auditadas: 10


,id,archivos,filas_totales,errores,columnas_territoriales,columnas_numericas
0,AMB-SAC,1,1313,0,cod_locali | localidad,
1,EDU-CUPOS,1,747,0,COD_LOCA,Aceleracio | CLASE_TIPO | Educacion_ | GENERO ...
2,FIN-RIVI,6,126,0,NombreLocalidad | NumeroLocalidad,Id_ | IndiceRespuesta | Numero | NumeroLocalid...
3,INFRA-PARQUES,1,139,0,Nombre Localidad,
4,MR,1,20,0,,LOCAREA | OBJECTID | SHAPE.AREA | SHAPE.LEN
5,POB-LOC,1,131502,0,CODIGO_LOCALIDAD | NOMBRE_LOCALIDAD,ANO | CODIGO_LOCALIDAD | EDAD | POBLACION
6,SAL-IPS,1,2900,0,,Id | OBJECTID | fax
7,SEG-CUADRANTES,1,599,0,,properties/PCUCOD_ENT | properties/PCUIULOCAL ...
8,TM-ESTACIONES,1,153,0,,biciparqueadero_estacion | capacidad_biciestac...
9,TM-PARADEROS,1,7653,0,localidad_,coordena_1 | coordenada | latitud_pa | localid...


### Diagnóstico del archivo de parques

Se inspecciona el archivo de infraestructura sin descartar filas, para identificar
su formato, codificación y separador antes de incorporarlo al modelo.

In [7]:
ruta_parques = rutas_unicas["INFRA-PARQUES"]
contenido = ruta_parques.read_bytes()

print("Ruta:", ruta_parques)
print("Tamaño en bytes:", len(contenido))
print("Primeros bytes:", repr(contenido[:150]))

if contenido.startswith(b"version https://git-lfs.github.com/spec/v1"):
    print("\nDIAGNÓSTICO: el archivo es un puntero de Git LFS, no el CSV real.")
elif contenido.startswith(b"PK\x03\x04"):
    print("\nDIAGNÓSTICO: parece un archivo Excel/ZIP aunque su extensión sea .csv.")
elif b"<html" in contenido[:500].lower() or b"<!doctype" in contenido[:500].lower():
    print("\nDIAGNÓSTICO: parece contenido HTML, no un CSV.")
elif len(contenido) == 0:
    print("\nDIAGNÓSTICO: el archivo está vacío.")

pruebas_lectura = [
    ("utf-8-sig", ",", "c"),
    ("utf-8-sig", ";", "c"),
    ("latin-1", ",", "c"),
    ("latin-1", ";", "c"),
    ("utf-16", "\t", "c"),
    ("latin-1", None, "python"),
]

print("\nPruebas de lectura:")
for encoding, separador, motor in pruebas_lectura:
    try:
        muestra = pd.read_csv(
            ruta_parques,
            encoding=encoding,
            sep=separador,
            engine=motor,
            nrows=5,
        )
        print(
            f"OK | encoding={encoding!r}, sep={separador!r}, "
            f"engine={motor!r} | columnas={muestra.columns.tolist()}"
        )
    except Exception as exc:
        detalle = str(exc).replace("\n", " ")[:250]
        print(
            f"FALLÓ | encoding={encoding!r}, sep={separador!r}, "
            f"engine={motor!r} | {type(exc).__name__}: {detalle}"
        )

Ruta: C:\Users\ADAN\DataJam_DataOlinguitos_Gen\data\processed\INFRAESTRUCTURA_ESPACIO_PUBLICO\5.-parques-idrd.csv
Tamaño en bytes: 7996
Primeros bytes: b'Codigo Parque;Nombre Parque;Nombre Localidad;Tipologia;Administracion\r\n01-012;La Vida;USAQU\xc3\x89N;ESTRUCTURANTE;IDRD\r\n01-023;Servita;USAQU\xc3\x89N;ESTRUCTURAN'

Pruebas de lectura:
OK | encoding='utf-8-sig', sep=',', engine='c' | columnas=['Codigo Parque;Nombre Parque;Nombre Localidad;Tipologia;Administracion']
OK | encoding='utf-8-sig', sep=';', engine='c' | columnas=['Codigo Parque', 'Nombre Parque', 'Nombre Localidad', 'Tipologia', 'Administracion']
OK | encoding='latin-1', sep=',', engine='c' | columnas=['Codigo Parque;Nombre Parque;Nombre Localidad;Tipologia;Administracion']
OK | encoding='latin-1', sep=';', engine='c' | columnas=['Codigo Parque', 'Nombre Parque', 'Nombre Localidad', 'Tipologia', 'Administracion']
FALLÓ | encoding='utf-16', sep='\t', engine='c' | UnicodeDecodeError: 'utf-16' codec can't decode bytes in po

### Auditoría semántica de la base territorial y demográfica

Se revisan los campos, tipos y valores de las fuentes MR y POB-LOC antes de
seleccionar la llave territorial, el área y los denominadores poblacionales.

In [8]:
def ficha_campos(df):
    registros = []

    for columna in df.columns:
        if columna == "geometry":
            continue

        serie_texto = df[columna].dropna().astype(str)
        ejemplos = serie_texto.drop_duplicates().head(3).tolist()

        registros.append(
            {
                "campo": columna,
                "tipo": str(df[columna].dtype),
                "nulos": int(df[columna].isna().sum()),
                "valores_unicos": int(serie_texto.nunique()),
                "ejemplos": " | ".join(ejemplos),
            }
        )

    return pd.DataFrame(registros)


mr = leer_archivo_modelo(rutas_unicas["MR"])
poblacion = leer_archivo_modelo(rutas_unicas["POB-LOC"])


print("FUENTE MR")
print("Filas:", len(mr))
print("Columnas:", len(mr.columns))
print("CRS:", getattr(mr, "crs", None))

if hasattr(mr, "geometry"):
    print("Geometrías:")
    print(mr.geometry.geom_type.value_counts(dropna=False))

display(ficha_campos(mr))


print("\nFUENTE POB-LOC")
print("Filas:", len(poblacion))
print("Columnas:", len(poblacion.columns))

display(ficha_campos(poblacion))

FUENTE MR
Filas: 20
Columnas: 8
CRS: EPSG:4326
Geometrías:
Polygon    20
Name: count, dtype: int64


,campo,tipo,nulos,valores_unicos,ejemplos
0,OBJECTID,int32,0,20,1 | 2 | 3
1,LOCNOMBRE,str,0,20,ANTONIO NARIÑO | TUNJUELITO | RAFAEL URIBE URIBE
2,LOCAADMINI,str,0,6,Acuerdo 117 de 2003 | Acuerdo 8 de 1977 | Acue...
3,LOCAREA,float64,0,20,4879543.3864294 | 9910939.74356609 | 13834084....
4,LOCCODIGO,str,0,20,15 | 06 | 18
5,SHAPE.AREA,float64,0,20,0.000397341362785198 | 0.000807032749259906 | ...
6,SHAPE.LEN,float64,0,20,0.108973020105687 | 0.21054198766811202 | 0.17...



FUENTE POB-LOC
Filas: 131502
Columnas: 8


,campo,tipo,nulos,valores_unicos,ejemplos
0,ANO,int64,0,31,2005 | 2006 | 2007
1,CODIGO_LOCALIDAD,int64,0,21,0 | 1 | 2
2,NOMBRE_LOCALIDAD,str,0,21,Bogotá | Usaquén | Chapinero
3,SEXO,str,0,2,Hombres | Mujeres
4,EDAD,int64,0,101,6 | 7 | 8
5,CURSODEVIDA,str,0,6,Infancia | Primera Infancia | Adolescencia
6,GRUPOEDAD,str,0,5,00 a 11 | 12 a 17 | 18 a 28
7,POBLACION,int64,0,14682,67184 | 68940 | 70568


### Construcción de la base territorial 2025

La unidad de análisis corresponde a las 20 localidades de Bogotá.

- La llave de integración es el código oficial de localidad.
- El área territorial se toma de `LOCAREA` y se convierte de m² a km².
- La población total corresponde a ambos sexos y todas las edades en 2025.
- La población escolar corresponde a ambos sexos entre 5 y 17 años.
- Se excluye el registro agregado de Bogotá identificado con el código 0.
- El área oficial se contrasta con el área calculada mediante la geometría
  proyectada en MAGNA-SIRGAS / Colombia Bogotá zone (`EPSG:3116`).

In [9]:
# Validación de la granularidad demográfica
clave_demografica = [
    "ANO",
    "CODIGO_LOCALIDAD",
    "SEXO",
    "EDAD",
]

duplicados_demografia = int(
    poblacion.duplicated(clave_demografica).sum()
)


# Preparación de la dimensión territorial
mr_base = mr.copy()

mr_base["codigo_localidad"] = (
    mr_base["LOCCODIGO"]
    .astype(str)
    .str.strip()
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

mr_base["localidad"] = (
    mr_base["LOCNOMBRE"]
    .astype(str)
    .str.strip()
)

# LOCAREA está expresada en metros cuadrados
mr_base["area_km2"] = (
    pd.to_numeric(mr_base["LOCAREA"], errors="raise")
    / 1_000_000
)

# Área calculada únicamente como control geográfico
mr_proyectado = mr.to_crs(epsg=3116)

mr_base["area_geometria_km2"] = (
    mr_proyectado.geometry.area
    / 1_000_000
)

mr_base["diferencia_area_pct"] = (
    (
        mr_base["area_geometria_km2"]
        - mr_base["area_km2"]
    ).abs()
    / mr_base["area_km2"]
    * 100
)


# Selección de población local para 2025
poblacion_2025_detalle = poblacion.loc[
    (poblacion["ANO"] == 2025)
    & (poblacion["CODIGO_LOCALIDAD"] != 0)
].copy()

poblacion_2025_detalle["codigo_localidad"] = (
    poblacion_2025_detalle["CODIGO_LOCALIDAD"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)


# Población total: ambos sexos y todas las edades
poblacion_total_2025 = (
    poblacion_2025_detalle
    .groupby("codigo_localidad", as_index=False)
    .agg(
        nombre_poblacion=("NOMBRE_LOCALIDAD", "first"),
        poblacion_2025=("POBLACION", "sum"),
    )
)


# Población objetivo para educación: edades de 5 a 17 años
poblacion_escolar_2025 = (
    poblacion_2025_detalle.loc[
        poblacion_2025_detalle["EDAD"].between(5, 17)
    ]
    .groupby("codigo_localidad", as_index=False)
    .agg(
        poblacion_5_17_2025=("POBLACION", "sum")
    )
)


poblacion_resumen_2025 = poblacion_total_2025.merge(
    poblacion_escolar_2025,
    on="codigo_localidad",
    how="left",
    validate="one_to_one",
)


# Verificación de correspondencia entre ambas fuentes
codigos_mr = set(mr_base["codigo_localidad"])
codigos_poblacion = set(
    poblacion_resumen_2025["codigo_localidad"]
)

print("Duplicados demográficos:", duplicados_demografia)
print("Filas demográficas de 2025:", len(poblacion_2025_detalle))
print("Códigos solamente en MR:", sorted(codigos_mr - codigos_poblacion))
print(
    "Códigos solamente en población:",
    sorted(codigos_poblacion - codigos_mr),
)


# Integración de la base territorial
base_territorial = mr_base[
    [
        "codigo_localidad",
        "localidad",
        "area_km2",
        "area_geometria_km2",
        "diferencia_area_pct",
        "geometry",
    ]
].merge(
    poblacion_resumen_2025,
    on="codigo_localidad",
    how="left",
    validate="one_to_one",
)

base_territorial = gpd.GeoDataFrame(
    base_territorial,
    geometry="geometry",
    crs=mr.crs,
).sort_values("codigo_localidad").reset_index(drop=True)


# Controles de aceptación
assert duplicados_demografia == 0
assert codigos_mr == codigos_poblacion
assert len(base_territorial) == 20
assert base_territorial["codigo_localidad"].is_unique
assert base_territorial["poblacion_2025"].notna().all()
assert base_territorial["poblacion_5_17_2025"].notna().all()
assert (base_territorial["area_km2"] > 0).all()
assert (base_territorial["poblacion_2025"] > 0).all()
assert (
    base_territorial["poblacion_5_17_2025"]
    <= base_territorial["poblacion_2025"]
).all()


print("\nBase territorial construida:", len(base_territorial), "localidades")
print(
    "Diferencia máxima entre áreas:",
    round(base_territorial["diferencia_area_pct"].max(), 4),
    "%",
)

display(
    base_territorial.drop(columns="geometry").round(
        {
            "area_km2": 3,
            "area_geometria_km2": 3,
            "diferencia_area_pct": 3,
        }
    )
)

Duplicados demográficos: 0
Filas demográficas de 2025: 4040
Códigos solamente en MR: []
Códigos solamente en población: []

Base territorial construida: 20 localidades
Diferencia máxima entre áreas: 0.0803 %


,codigo_localidad,localidad,area_km2,area_geometria_km2,diferencia_area_pct,nombre_poblacion,poblacion_2025,poblacion_5_17_2025
0,01,USAQUEN,65.201,65.149,0.080,Usaquén,586286,80440
1,02,CHAPINERO,38.009,37.978,0.080,Chapinero,161162,19753
2,03,SANTA FE,45.171,45.134,0.080,Santa Fe,113315,18196
3,04,SAN CRISTOBAL,49.099,49.059,0.080,San Cristóbal,394686,71189
4,05,USME,215.067,214.895,0.080,Usme,399153,79885
5,06,TUNJUELITO,9.911,9.903,0.080,Tunjuelito,175399,28636
6,07,BOSA,23.933,23.914,0.080,Bosa,768002,150775
7,08,KENNEDY,38.590,38.559,0.080,Kennedy,1103801,189208
8,09,FONTIBON,33.281,33.254,0.080,Fontibón,385455,55834
9,10,ENGATIVA,35.881,35.852,0.080,Engativá,831165,122474


### Auditoría semántica de educación y salud

Se inspecciona la granularidad de las fuentes antes de calcular indicadores.

Para educación se debe identificar el campo de oferta total y comprobar que el
código de localidad corresponda con la dimensión territorial.

Para salud se debe identificar la unidad institucional que puede contarse sin
duplicación. La localidad se asignará mediante cruce espacial con los polígonos
territoriales, sin interpretar automáticamente cada fila como una IPS distinta.

In [10]:
educacion = leer_archivo_modelo(
    rutas_unicas["EDU-CUPOS"]
)

salud = leer_archivo_modelo(
    rutas_unicas["SAL-IPS"]
)


def campos_relevantes(df, patron):
    ficha = ficha_campos(df)

    campos_numericos = {
        columna
        for columna in df.columns
        if columna != "geometry"
        and pd.api.types.is_numeric_dtype(df[columna])
    }

    seleccion = (
        ficha["campo"].astype(str).str.contains(
            patron,
            case=False,
            regex=True,
            na=False,
        )
        | ficha["campo"].isin(campos_numericos)
    )

    return ficha.loc[seleccion].reset_index(drop=True)


patron_educacion = (
    r"loc|cod|dane|instit|sede|nom|cupo|oferta|total|"
    r"nivel|grado|jornada|edad|genero"
)

patron_salud = (
    r"loc|cod|ips|prest|sede|nom|razon|nit|serv|urg|"
    r"nivel|direccion|object|^id$"
)


# -------------------------------------------------
# Educación
# -------------------------------------------------
print("FUENTE EDU-CUPOS")
print("Filas:", len(educacion))
print("Columnas:", len(educacion.columns))
print("CRS:", getattr(educacion, "crs", None))

if hasattr(educacion, "geometry"):
    print("Geometrías:")
    print(educacion.geometry.geom_type.value_counts(dropna=False))
    print("Geometrías nulas:", int(educacion.geometry.isna().sum()))
    print("Geometrías vacías:", int(educacion.geometry.is_empty.sum()))
    print("Geometrías inválidas:", int((~educacion.geometry.is_valid).sum()))
    print(
        "Geometrías repetidas:",
        int(educacion.geometry.to_wkb().duplicated().sum()),
    )

codigos_educacion = set(
    pd.to_numeric(
        educacion["COD_LOCA"],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

print(
    "Códigos educativos no presentes en MR:",
    sorted(codigos_educacion - codigos_mr),
)
print(
    "Localidades de MR sin registros educativos:",
    sorted(codigos_mr - codigos_educacion),
)

display(
    campos_relevantes(
        educacion,
        patron_educacion,
    )
)


# -------------------------------------------------
# Salud
# -------------------------------------------------
print("\nFUENTE SAL-IPS")
print("Filas:", len(salud))
print("Columnas:", len(salud.columns))
print("CRS:", getattr(salud, "crs", None))

if hasattr(salud, "geometry"):
    print("Geometrías:")
    print(salud.geometry.geom_type.value_counts(dropna=False))
    print("Geometrías nulas:", int(salud.geometry.isna().sum()))
    print("Geometrías vacías:", int(salud.geometry.is_empty.sum()))
    print("Geometrías inválidas:", int((~salud.geometry.is_valid).sum()))
    print(
        "Geometrías repetidas:",
        int(salud.geometry.to_wkb().duplicated().sum()),
    )

display(
    campos_relevantes(
        salud,
        patron_salud,
    )
)


# Cruce espacial preliminar: todavía no agrega ni cuenta IPS
salud_geo = salud.to_crs(mr.crs)

territorios_cruce = mr_base[
    [
        "codigo_localidad",
        "localidad",
        "geometry",
    ]
].copy()

salud_localizada = gpd.sjoin(
    salud_geo,
    territorios_cruce,
    how="left",
    predicate="within",
)

print("\nCONTROL DEL CRUCE ESPACIAL DE SALUD")
print("Filas originales:", len(salud))
print("Filas después del cruce:", len(salud_localizada))
print(
    "Registros asignados:",
    int(salud_localizada["codigo_localidad"].notna().sum()),
)
print(
    "Registros sin localidad:",
    int(salud_localizada["codigo_localidad"].isna().sum()),
)

FUENTE EDU-CUPOS
Filas: 747
Columnas: 14
CRS: EPSG:3857
Geometrías:
Point    747
Name: count, dtype: int64
Geometrías nulas: 0
Geometrías vacías: 0
Geometrías inválidas: 0
Geometrías repetidas: 6
Códigos educativos no presentes en MR: []
Localidades de MR sin registros educativos: []


,campo,tipo,nulos,valores_unicos,ejemplos
0,NOMBRE_EST,str,0,408,COLEGIO AQUILEO PARRA (IED) | COLEGIO AGUSTIN ...
1,GENERO,int32,0,2,5 | 1
2,COD_LOCA,str,0,20,01 | 02 | 03
3,CLASE_TIPO,int32,0,2,1 | 2
4,OPreescola,int32,0,268,170 | 0 | 40
5,OPrimaria,int32,0,391,1115 | 317 | 200
6,OSecundari,int32,0,288,1045 | 599 | 0
7,OMedia,int32,0,237,313 | 262 | 0
8,OTotal,int32,0,618,2920 | 1500 | 240
9,Aceleracio,int32,0,59,0 | 74 | 50



FUENTE SAL-IPS
Filas: 2900
Columnas: 28
CRS: EPSG:4326
Geometrías:
Point    2900
Name: count, dtype: int64
Geometrías nulas: 0
Geometrías vacías: 0
Geometrías inválidas: 0
Geometrías repetidas: 782


,campo,tipo,nulos,valores_unicos,ejemplos
0,OBJECTID,int64,0,2900,1 | 2 | 3
1,Id,float64,0,2900,1.0 | 2.0 | 3.0
2,codigo_pre,str,0,1527,1100100032 | 1100100103 | 1100100130
3,nombre_pre,str,0,1527,COOPERATIVA PARA LA SALUD ORAL ORALCOOP | INST...
4,nombre,str,0,2676,COOPERATIVA PARA LA SALUD ORAL ORALCOOP | INST...
5,direccion,str,0,2896,KR 64 # 100 55 | CLL 43 # 25 - 61 | DIAGONAL 1...
6,fax,float64,0,743,5330579.0 | 0.0 | 4425296.0
7,nivel,str,0,2,| 3
8,sede_princ,str,0,2,SI | NO



CONTROL DEL CRUCE ESPACIAL DE SALUD
Filas originales: 2900
Filas después del cruce: 2900
Registros asignados: 2900
Registros sin localidad: 0


### Auditoría de granularidad de educación y salud
Antes de agregar los indicadores se comprueba la unidad real de cada registro.

En educación se analiza la relación entre establecimiento, ubicación y oferta de
cupos. En salud se distingue entre prestador jurídico, sede física y registro
individual, evitando contar identificadores técnicos como si fueran IPS distintas.

In [11]:
def clave_geometrica(serie_geometrica):
    return serie_geometrica.apply(
        lambda geometria: (
            geometria.wkb_hex
            if geometria is not None and not geometria.is_empty
            else None
        )
    )


# =================================================
# EDUCACIÓN
# =================================================
educacion_audit = educacion.copy()
educacion_audit["_geometria"] = clave_geometrica(
    educacion_audit.geometry
)

columnas_originales_edu = [
    columna
    for columna in educacion.columns
    if columna != "geometry"
]

duplicados_exactos_edu = int(
    educacion_audit.duplicated(
        subset=columnas_originales_edu + ["_geometria"]
    ).sum()
)

componentes_oferta = [
    "OPreescola",
    "OPrimaria",
    "OSecundari",
    "OMedia",
    "Aceleracio",
    "Educacion_",
]

educacion_audit["_suma_componentes"] = (
    educacion_audit[componentes_oferta].sum(axis=1)
)

educacion_audit["_diferencia_total"] = (
    educacion_audit["OTotal"]
    - educacion_audit["_suma_componentes"]
)

resumen_dane = (
    educacion_audit
    .groupby("DANE12_EST", as_index=False)
    .agg(
        filas=("DANE12_EST", "size"),
        nombres=("NOMBRE_EST", "nunique"),
        localidades=("COD_LOCA", "nunique"),
        geometrias=("_geometria", "nunique"),
        ototal_min=("OTotal", "min"),
        ototal_max=("OTotal", "max"),
        ototal_suma=("OTotal", "sum"),
    )
)

print("AUDITORÍA DE EDUCACIÓN")
print("Columnas:", educacion.columns.tolist())
print("Duplicados exactos:", duplicados_exactos_edu)
print(
    "Filas donde OTotal difiere de la suma de componentes:",
    int((educacion_audit["_diferencia_total"] != 0).sum()),
)
print(
    "DANE con más de una fila:",
    int((resumen_dane["filas"] > 1).sum()),
)
print(
    "DANE ubicados en más de un punto:",
    int((resumen_dane["geometrias"] > 1).sum()),
)
print(
    "DANE presentes en más de una localidad:",
    int((resumen_dane["localidades"] > 1).sum()),
)

print("\nDistribución de filas por código DANE:")
display(
    resumen_dane["filas"]
    .value_counts()
    .sort_index()
    .rename_axis("filas_por_dane")
    .to_frame("cantidad_dane")
)

print("\nEjemplos de códigos DANE repetidos:")
display(
    resumen_dane.loc[
        resumen_dane["filas"] > 1
    ].head(10)
)


# =================================================
# SALUD
# =================================================
salud_audit = salud.copy()
salud_audit["_geometria"] = clave_geometrica(
    salud_audit.geometry
)

# El cruce conservó exactamente una fila por registro
salud_audit["codigo_localidad"] = (
    salud_localizada["codigo_localidad"]
)

salud_audit["localidad"] = (
    salud_localizada["localidad"]
)

# Duplicados sustantivos excluyendo identificadores técnicos
columnas_salud_sin_ids = [
    columna
    for columna in salud.columns
    if columna not in {"geometry", "OBJECTID", "Id"}
]

salud_contenido_sin_ids = (
    salud_audit[columnas_salud_sin_ids]
    .astype("string")
    .copy()
)

salud_contenido_sin_ids["_geometria"] = (
    salud_audit["_geometria"]
)

duplicados_salud_sin_ids = int(
    salud_contenido_sin_ids.duplicated().sum()
)


# Clave auditable de posible sede física
componentes_clave_sede = pd.DataFrame(
    {
        "codigo_prestador": (
            salud_audit["codigo_pre"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("<NA>")
        ),
        "nombre_sede": (
            salud_audit["nombre"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("<NA>")
        ),
        "direccion": (
            salud_audit["direccion"]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("<NA>")
        ),
        "geometria": (
            salud_audit["_geometria"]
            .astype("string")
            .fillna("<NA>")
        ),
    }
)

salud_audit["_clave_sede"] = (
    pd.util.hash_pandas_object(
        componentes_clave_sede,
        index=False,
    )
    .astype("uint64")
)


resumen_sedes_salud = (
    salud_audit
    .groupby("_clave_sede", as_index=False)
    .agg(
        filas=("Id", "size"),
        ids=("Id", "nunique"),
        codigo_prestador=("codigo_pre", "first"),
        nombre_prestador=("nombre_pre", "first"),
        nombre_sede=("nombre", "first"),
        direccion=("direccion", "first"),
        codigo_localidad=("codigo_localidad", "first"),
    )
)

localidades_por_prestador = (
    salud_audit
    .groupby("codigo_pre")["codigo_localidad"]
    .nunique()
)

print("\nAUDITORÍA DE SALUD")
print("Duplicados de contenido excluyendo Id y OBJECTID:",
      duplicados_salud_sin_ids)
print("Prestadores únicos:",
      salud_audit["codigo_pre"].nunique())
print("Claves de sede únicas:",
      salud_audit["_clave_sede"].nunique())
print(
    "Claves de sede con más de una fila:",
    int((resumen_sedes_salud["filas"] > 1).sum()),
)
print(
    "Prestadores presentes en más de una localidad:",
    int((localidades_por_prestador > 1).sum()),
)

print("\nDistribución de filas por posible sede:")
display(
    resumen_sedes_salud["filas"]
    .value_counts()
    .sort_index()
    .rename_axis("filas_por_sede")
    .to_frame("cantidad_sedes")
)

print("\nEjemplos de posibles sedes repetidas:")
display(
    resumen_sedes_salud.loc[
        resumen_sedes_salud["filas"] > 1
    ].head(10)
)

AUDITORÍA DE EDUCACIÓN
Columnas: ['NOMBRE_EST', 'GENERO', 'COD_LOCA', 'CLASE_TIPO', 'FECHA', 'OPreescola', 'OPrimaria', 'OSecundari', 'OMedia', 'OTotal', 'Aceleracio', 'DANE12_EST', 'Educacion_', 'geometry']
Duplicados exactos: 0
Filas donde OTotal difiere de la suma de componentes: 0
DANE con más de una fila: 208
DANE ubicados en más de un punto: 207
DANE presentes en más de una localidad: 1

Distribución de filas por código DANE:


,cantidad_dane
filas_por_dane,
1,204
2,122
3,62
4,20
5,2
9,1
14,1



Ejemplos de códigos DANE repetidos:


,DANE12_EST,filas,nombres,localidades,geometrias,ototal_min,ototal_max,ototal_suma
0,111001000078,3,1,1,3,274,524,1092
3,111001000272,2,1,1,2,1419,2004,3423
4,111001000612,2,1,1,2,186,530,716
5,111001001121,4,1,1,4,106,468,1003
6,111001001279,3,1,1,3,205,1199,1965
8,111001002330,2,1,1,2,942,1312,2254
9,111001002909,3,1,1,3,425,4109,5370
11,111001006483,2,1,1,2,80,2691,2771
12,111001008389,4,1,1,4,161,732,1324
13,111001009148,2,1,1,2,631,682,1313



AUDITORÍA DE SALUD
Duplicados de contenido excluyendo Id y OBJECTID: 0
Prestadores únicos: 1527
Claves de sede únicas: 2900
Claves de sede con más de una fila: 0
Prestadores presentes en más de una localidad: 252

Distribución de filas por posible sede:


,cantidad_sedes
filas_por_sede,
1,2900



Ejemplos de posibles sedes repetidas:


,_clave_sede,filas,ids,codigo_prestador,nombre_prestador,nombre_sede,direccion,codigo_localidad


### Indicadores territoriales de educación y salud

**Educación**

La oferta total corresponde a la suma de `OTotal` de cada registro localizado.
No se deduplica por código DANE porque un establecimiento puede tener varias
ubicaciones con ofertas diferentes.

\[
\text{Cupos por 1.000 habitantes de 5 a 17 años}
=
\frac{\sum OTotal}{Población_{5-17}} \times 1.000
\]

**Salud**

La unidad contada es la sede registrada, definida mediante prestador, nombre,
dirección y ubicación. No se interpreta como número de hospitales ni se
deduplica únicamente por prestador.

\[
\text{Sedes IPS por 10.000 habitantes}
=
\frac{\text{Sedes IPS registradas}}{Población total} \times 10.000
\]

En ambos indicadores, una mayor disponibilidad representa menor prioridad
territorial. Esta dirección se aplicará posteriormente durante la normalización.

#### Criterio de territorialización educativa

`COD_LOCA` se utiliza como llave territorial canónica porque es un atributo
explícito de la fuente y presenta cobertura completa de las 20 localidades.

La geometría se utiliza como control de calidad. Se identificaron tres
discrepancias entre el código y el polígono, además de un punto ubicado fuera
de los polígonos. Estos registros se conservan en la localidad declarada por
la fuente y quedan documentados como observaciones geográficas. No se realizan
correcciones manuales por establecimiento.

In [12]:
# =================================================
# CONTROL ESPACIAL DE EDUCACIÓN
# =================================================
educacion_geo = educacion.to_crs(mr.crs).copy()

educacion_geo["codigo_localidad_fuente"] = (
    pd.to_numeric(
        educacion_geo["COD_LOCA"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

territorios_educacion = mr_base[
    ["codigo_localidad", "geometry"]
].rename(
    columns={
        "codigo_localidad": "codigo_localidad_espacial"
    }
)

educacion_localizada = gpd.sjoin(
    educacion_geo,
    territorios_educacion,
    how="left",
    predicate="within",
)

sin_localidad_espacial_edu = int(
    educacion_localizada[
        "codigo_localidad_espacial"
    ].isna().sum()
)

diferencias_localidad_edu = (
    educacion_localizada[
        "codigo_localidad_espacial"
    ].notna()
    & (
        educacion_localizada["codigo_localidad_fuente"]
        != educacion_localizada["codigo_localidad_espacial"]
    )
)

print("CONTROL ESPACIAL DE EDUCACIÓN")
print("Filas originales:", len(educacion))
print("Filas después del cruce:", len(educacion_localizada))
print(
    "Registros sin localidad espacial:",
    sin_localidad_espacial_edu,
)
print(
    "Diferencias entre COD_LOCA y geometría:",
    int(diferencias_localidad_edu.sum()),
)
print(
    "Valores de FECHA:",
    educacion["FECHA"].drop_duplicates().astype(str).tolist(),
)


# Clasificación del control geográfico
educacion_localizada["estado_control_territorial"] = "coincide"

educacion_localizada.loc[
    educacion_localizada[
        "codigo_localidad_espacial"
    ].isna(),
    "estado_control_territorial",
] = "fuera_de_poligonos"

educacion_localizada.loc[
    diferencias_localidad_edu,
    "estado_control_territorial",
] = "discrepancia_codigo_geometria"


# Conservación de las observaciones para trazabilidad
observaciones_territoriales_edu = (
    educacion_localizada.loc[
        educacion_localizada[
            "estado_control_territorial"
        ] != "coincide",
        [
            "DANE12_EST",
            "NOMBRE_EST",
            "COD_LOCA",
            "codigo_localidad_fuente",
            "codigo_localidad_espacial",
            "OTotal",
            "estado_control_territorial",
        ],
    ]
    .copy()
)


codigos_educacion_fuente = set(
    educacion_localizada[
        "codigo_localidad_fuente"
    ].dropna()
)


# Controles que sí deben cumplirse
assert len(educacion_localizada) == len(educacion)
assert educacion_localizada[
    "codigo_localidad_fuente"
].notna().all()
assert codigos_educacion_fuente == codigos_mr


print("\nResultado del control territorial:")
print(
    educacion_localizada[
        "estado_control_territorial"
    ].value_counts()
)


# =================================================
# AGREGACIÓN DE EDUCACIÓN
# =================================================
indicador_educacion = (
    educacion_localizada
    .groupby(
        "codigo_localidad_fuente",
        as_index=False,
    )
    .agg(
        registros_oferta=("OTotal", "size"),
        establecimientos_dane=(
            "DANE12_EST",
            "nunique",
        ),
        oferta_total_cupos=("OTotal", "sum"),
    )
    .rename(
        columns={
            "codigo_localidad_fuente":
                "codigo_localidad"
        }
    )
)


# =================================================
# AGREGACIÓN DE SALUD
# =================================================
indicador_salud = (
    salud_audit
    .groupby(
        "codigo_localidad",
        as_index=False,
    )
    .agg(
        registros_salud=("Id", "size"),
        sedes_ips_registradas=(
            "_clave_sede",
            "nunique",
        ),
        prestadores_unicos_localidad=(
            "codigo_pre",
            "nunique",
        ),
    )
)


# =================================================
# INTEGRACIÓN CON LA BASE TERRITORIAL
# =================================================
base_indicadores = (
    base_territorial
    .merge(
        indicador_educacion,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
    .merge(
        indicador_salud,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
)

base_indicadores = gpd.GeoDataFrame(
    base_indicadores,
    geometry="geometry",
    crs=base_territorial.crs,
)


# Tasas territoriales
base_indicadores["cupos_por_1000_pob_5_17"] = (
    base_indicadores["oferta_total_cupos"]
    / base_indicadores["poblacion_5_17_2025"]
    * 1_000
)

base_indicadores["sedes_ips_por_10000_hab"] = (
    base_indicadores["sedes_ips_registradas"]
    / base_indicadores["poblacion_2025"]
    * 10_000
)


# =================================================
# CONTROLES DE ACEPTACIÓN
# =================================================
columnas_control = [
    "oferta_total_cupos",
    "sedes_ips_registradas",
    "cupos_por_1000_pob_5_17",
    "sedes_ips_por_10000_hab",
]

assert len(base_indicadores) == 20
assert base_indicadores["codigo_localidad"].is_unique
assert base_indicadores[columnas_control].notna().all().all()
assert base_indicadores["oferta_total_cupos"].sum() == educacion["OTotal"].sum()
assert base_indicadores["registros_oferta"].sum() == len(educacion)
assert base_indicadores["registros_salud"].sum() == len(salud)
assert base_indicadores["sedes_ips_registradas"].sum() == 2900


print("\nINDICADORES CONSTRUIDOS")
print("Localidades:", len(base_indicadores))
print(
    "Registros educativos agregados:",
    int(base_indicadores["registros_oferta"].sum()),
)
print(
    "Oferta total de cupos:",
    int(base_indicadores["oferta_total_cupos"].sum()),
)
print(
    "Sedes IPS agregadas:",
    int(base_indicadores["sedes_ips_registradas"].sum()),
)

display(
    base_indicadores[
        [
            "codigo_localidad",
            "localidad",
            "poblacion_2025",
            "poblacion_5_17_2025",
            "oferta_total_cupos",
            "cupos_por_1000_pob_5_17",
            "sedes_ips_registradas",
            "sedes_ips_por_10000_hab",
        ]
    ].round(2)
)

CONTROL ESPACIAL DE EDUCACIÓN
Filas originales: 747
Filas después del cruce: 747
Registros sin localidad espacial: 1
Diferencias entre COD_LOCA y geometría: 3
Valores de FECHA: ['2025-03-31']

Resultado del control territorial:
estado_control_territorial
coincide                         743
discrepancia_codigo_geometria      3
fuera_de_poligonos                 1
Name: count, dtype: int64

INDICADORES CONSTRUIDOS
Localidades: 20
Registros educativos agregados: 747
Oferta total de cupos: 818542
Sedes IPS agregadas: 2900


,codigo_localidad,localidad,poblacion_2025,poblacion_5_17_2025,oferta_total_cupos,cupos_por_1000_pob_5_17,sedes_ips_registradas,sedes_ips_por_10000_hab
0,01,USAQUEN,586286,80440,25413,315.92,560,9.55
1,02,CHAPINERO,161162,19753,3808,192.78,511,31.71
2,03,SANTA FE,113315,18196,10144,557.49,78,6.88
3,04,SAN CRISTOBAL,394686,71189,53625,753.28,31,0.79
4,05,USME,399153,79885,74680,934.84,24,0.60
5,06,TUNJUELITO,175399,28636,31433,1097.67,33,1.88
6,07,BOSA,768002,150775,115600,766.71,46,0.60
7,08,KENNEDY,1103801,189208,110581,584.44,193,1.75
8,09,FONTIBON,385455,55834,28990,519.22,137,3.55
9,10,ENGATIVA,831165,122474,67938,554.71,172,2.07


In [13]:
# Registros con ausencia o discrepancia territorial
problemas_edu = educacion_localizada.loc[
    educacion_localizada[
        "codigo_localidad_espacial"
    ].isna()
    | (
        educacion_localizada[
            "codigo_localidad_fuente"
        ]
        != educacion_localizada[
            "codigo_localidad_espacial"
        ]
    )
].copy()

problemas_edu = problemas_edu.drop(
    columns=["index_right"],
    errors="ignore",
)

problemas_edu["indice_fuente"] = problemas_edu.index


# Nombres asociados a los códigos
nombres_por_codigo = (
    mr_base
    .set_index("codigo_localidad")["localidad"]
    .to_dict()
)

problemas_edu["localidad_fuente"] = (
    problemas_edu["codigo_localidad_fuente"]
    .map(nombres_por_codigo)
)

problemas_edu["localidad_espacial"] = (
    problemas_edu["codigo_localidad_espacial"]
    .map(nombres_por_codigo)
)


# Coordenadas para trazabilidad
problemas_edu["longitud"] = problemas_edu.geometry.x
problemas_edu["latitud"] = problemas_edu.geometry.y


# Proyección métrica para calcular distancias
problemas_edu_proyectados = problemas_edu.to_crs(
    epsg=3116
)

territorios_proyectados = (
    mr_base
    .to_crs(epsg=3116)[
        [
            "codigo_localidad",
            "localidad",
            "geometry",
        ]
    ]
)

territorios_cercanos = territorios_proyectados.rename(
    columns={
        "codigo_localidad": "codigo_localidad_cercana",
        "localidad": "localidad_cercana",
    }
)


# Localidad más cercana a cada punto problemático
auditoria_territorial_edu = gpd.sjoin_nearest(
    problemas_edu_proyectados,
    territorios_cercanos,
    how="left",
    distance_col="distancia_localidad_cercana_m",
)


# Distancia entre el punto y la localidad declarada en COD_LOCA
poligonos_por_codigo = (
    territorios_proyectados
    .set_index("codigo_localidad")
    .geometry
    .to_dict()
)

auditoria_territorial_edu[
    "distancia_localidad_fuente_m"
] = [
    geometria.distance(
        poligonos_por_codigo[codigo]
    )
    for geometria, codigo in zip(
        auditoria_territorial_edu.geometry,
        auditoria_territorial_edu[
            "codigo_localidad_fuente"
        ],
    )
]


columnas_auditoria = [
    "indice_fuente",
    "DANE12_EST",
    "NOMBRE_EST",
    "FECHA",
    "OTotal",
    "codigo_localidad_fuente",
    "localidad_fuente",
    "codigo_localidad_espacial",
    "localidad_espacial",
    "codigo_localidad_cercana",
    "localidad_cercana",
    "distancia_localidad_cercana_m",
    "distancia_localidad_fuente_m",
    "longitud",
    "latitud",
]

display(
    auditoria_territorial_edu[
        columnas_auditoria
    ].round(
        {
            "distancia_localidad_cercana_m": 2,
            "distancia_localidad_fuente_m": 2,
            "longitud": 6,
            "latitud": 6,
        }
    )
)

,indice_fuente,DANE12_EST,NOMBRE_EST,FECHA,OTotal,codigo_localidad_fuente,localidad_fuente,codigo_localidad_espacial,localidad_espacial,codigo_localidad_cercana,localidad_cercana,distancia_localidad_cercana_m,distancia_localidad_fuente_m,longitud,latitud
182,182,211850001074,COLEGIO RURAL LAS MERCEDES (CED),2025-03-31,44,05,USME,19,CIUDAD BOLIVAR,19,CIUDAD BOLIVAR,0.00,401.49,-74.17622,4.39225
244,244,111001107883,COLEGIO DEBORA ARANGO PEREZ (IED),2025-03-31,3009,07,BOSA,08,KENNEDY,08,KENNEDY,0.00,30.88,-74.18348,4.61883
590,590,111001013323,COLEGIO INTEGRADA LA CANDELARIA (IED),2025-03-31,329,17,CANDELARIA,03,SANTA FE,03,SANTA FE,0.00,0.69,-74.06800,4.60151
727,727,211001076346,COLEGIO CAMPESTRE JAIME GARZON (IED),2025-03-31,31,20,SUMAPAZ,NaN,NaN,20,SUMAPAZ,219.33,219.33,-74.14300,4.24000


### Refinamiento del indicador educativo y revisión de valores atípicos

El indicador educativo principal utiliza la oferta de preescolar, primaria,
secundaria y media, debido a su correspondencia con la población objetivo de
5 a 17 años.

Las categorías `Aceleracio` y `Educacion_` se conservan como modalidades
complementarias, pero no se incorporan al indicador principal porque la fuente
no desagrega explícitamente su población objetivo.

Los valores atípicos territoriales se documentan, pero no se eliminan ni
recortan. Posteriormente se evaluará su efecto mediante análisis de sensibilidad.

In [14]:
# Componentes con población objetivo escolar identificable
componentes_oferta_regular = [
    "OPreescola",
    "OPrimaria",
    "OSecundari",
    "OMedia",
]

educacion_refinada = educacion_localizada.copy()

educacion_refinada["oferta_regular_cupos"] = (
    educacion_refinada[
        componentes_oferta_regular
    ].sum(axis=1)
)

educacion_refinada[
    "oferta_modalidades_complementarias"
] = (
    educacion_refinada["Aceleracio"]
    + educacion_refinada["Educacion_"]
)

# Verificación contra el total informado por la fuente
assert (
    educacion_refinada["oferta_regular_cupos"]
    + educacion_refinada[
        "oferta_modalidades_complementarias"
    ]
    == educacion_refinada["OTotal"]
).all()


oferta_educativa_refinada = (
    educacion_refinada
    .groupby(
        "codigo_localidad_fuente",
        as_index=False,
    )
    .agg(
        oferta_regular_cupos=(
            "oferta_regular_cupos",
            "sum",
        ),
        oferta_modalidades_complementarias=(
            "oferta_modalidades_complementarias",
            "sum",
        ),
    )
    .rename(
        columns={
            "codigo_localidad_fuente":
                "codigo_localidad"
        }
    )
)


# Permite volver a ejecutar la celda sin duplicar columnas
base_indicadores = base_indicadores.drop(
    columns=[
        "oferta_regular_cupos",
        "oferta_modalidades_complementarias",
        "oferta_regular_por_1000_pob_5_17",
        "oferta_total_por_1000_pob_5_17",
        "participacion_complementaria_pct",
    ],
    errors="ignore",
)

base_indicadores = base_indicadores.merge(
    oferta_educativa_refinada,
    on="codigo_localidad",
    how="left",
    validate="one_to_one",
)

base_indicadores = gpd.GeoDataFrame(
    base_indicadores,
    geometry="geometry",
    crs=base_territorial.crs,
)


# Indicador principal
base_indicadores[
    "oferta_regular_por_1000_pob_5_17"
] = (
    base_indicadores["oferta_regular_cupos"]
    / base_indicadores["poblacion_5_17_2025"]
    * 1_000
)

# Indicador diagnóstico con OTotal
base_indicadores[
    "oferta_total_por_1000_pob_5_17"
] = (
    base_indicadores["oferta_total_cupos"]
    / base_indicadores["poblacion_5_17_2025"]
    * 1_000
)

base_indicadores[
    "participacion_complementaria_pct"
] = (
    base_indicadores[
        "oferta_modalidades_complementarias"
    ]
    / base_indicadores["oferta_total_cupos"]
    * 100
)


# Control de integridad
assert (
    base_indicadores["oferta_regular_cupos"]
    + base_indicadores[
        "oferta_modalidades_complementarias"
    ]
    == base_indicadores["oferta_total_cupos"]
).all()


# =================================================
# AUDITORÍA IQR
# =================================================
def auditoria_iqr(data, indicadores):
    resumen = []
    atipicos = []

    for indicador in indicadores:
        serie = data[indicador]

        q1 = serie.quantile(0.25)
        mediana = serie.median()
        q3 = serie.quantile(0.75)
        iqr = q3 - q1

        limite_inferior = q1 - 1.5 * iqr
        limite_superior = q3 + 1.5 * iqr

        mascara = (
            (serie < limite_inferior)
            | (serie > limite_superior)
        )

        resumen.append(
            {
                "indicador": indicador,
                "minimo": serie.min(),
                "q1": q1,
                "mediana": mediana,
                "q3": q3,
                "maximo": serie.max(),
                "limite_inferior": limite_inferior,
                "limite_superior": limite_superior,
                "cantidad_atipicos": int(mascara.sum()),
                "asimetria": serie.skew(),
            }
        )

        for _, fila in data.loc[
            mascara,
            ["localidad", indicador],
        ].iterrows():
            atipicos.append(
                {
                    "indicador": indicador,
                    "localidad": fila["localidad"],
                    "valor": fila[indicador],
                    "tipo": (
                        "inferior"
                        if fila[indicador] < limite_inferior
                        else "superior"
                    ),
                }
            )

    return (
        pd.DataFrame(resumen),
        pd.DataFrame(atipicos),
    )


resumen_distribuciones, valores_atipicos = auditoria_iqr(
    base_indicadores,
    [
        "oferta_regular_por_1000_pob_5_17",
        "sedes_ips_por_10000_hab",
    ],
)


print("CONTROL DEL REFINAMIENTO EDUCATIVO")
print(
    "Oferta total:",
    int(base_indicadores["oferta_total_cupos"].sum()),
)
print(
    "Oferta regular:",
    int(base_indicadores["oferta_regular_cupos"].sum()),
)
print(
    "Modalidades complementarias:",
    int(
        base_indicadores[
            "oferta_modalidades_complementarias"
        ].sum()
    ),
)

print("\nRESUMEN DE DISTRIBUCIONES")
display(resumen_distribuciones.round(2))

print("\nVALORES ATÍPICOS DOCUMENTADOS")
display(valores_atipicos.round(2))

CONTROL DEL REFINAMIENTO EDUCATIVO
Oferta total: 818542
Oferta regular: 772137
Modalidades complementarias: 46405

RESUMEN DE DISTRIBUCIONES


,indicador,minimo,q1,mediana,q3,maximo,limite_inferior,limite_superior,cantidad_atipicos,asimetria
0,oferta_regular_por_1000_pob_5_17,172.73,489.91,676.37,785.11,3340.34,47.11,1227.91,2,3.08
1,sedes_ips_por_10000_hab,0.60,1.72,3.70,8.93,31.71,-9.11,19.76,1,2.19



VALORES ATÍPICOS DOCUMENTADOS


,indicador,localidad,valor,tipo
0,oferta_regular_por_1000_pob_5_17,CANDELARIA,1572.39,superior
1,oferta_regular_por_1000_pob_5_17,SUMAPAZ,3340.34,superior
2,sedes_ips_por_10000_hab,CHAPINERO,31.71,superior


### Auditoría semántica y espacial de movilidad

Se inspeccionan las estaciones troncales y los paraderos zonales antes de
calcular sus densidades territoriales.

La geometría se cruza con la dimensión territorial para asignar cada elemento
a una localidad. Antes de contar se comprueban identificadores, duplicados,
geometrías inválidas y registros ubicados fuera de los polígonos.

In [15]:
estaciones = leer_archivo_modelo(
    rutas_unicas["TM-ESTACIONES"]
)

paraderos = leer_archivo_modelo(
    rutas_unicas["TM-PARADEROS"]
)


def duplicados_exactos_geograficos(df):
    datos = (
        df.drop(columns="geometry")
        .astype("string")
        .copy()
    )

    datos["_geometria"] = clave_geometrica(
        df.geometry
    )

    return int(datos.duplicated().sum())


def imprimir_control_geografico(nombre, df):
    print(nombre)
    print("Filas:", len(df))
    print("Columnas:", len(df.columns))
    print("CRS:", getattr(df, "crs", None))
    print("Tipos de geometría:")
    print(df.geometry.geom_type.value_counts(dropna=False))
    print("Geometrías nulas:", int(df.geometry.isna().sum()))
    print("Geometrías vacías:", int(df.geometry.is_empty.sum()))
    print("Geometrías inválidas:", int((~df.geometry.is_valid).sum()))
    print(
        "Geometrías repetidas:",
        int(clave_geometrica(df.geometry).duplicated().sum()),
    )
    print(
        "Duplicados exactos:",
        duplicados_exactos_geograficos(df),
    )


patron_estaciones = (
    r"estacion|nombre|nom|codigo|cod|^id$|object|"
    r"troncal|tipo|linea|portal|capacidad|bici"
)

patron_paraderos = (
    r"paradero|nombre|nom|codigo|cod|^id$|object|"
    r"sitp|zona|tipo|direccion|local|latitud|longitud"
)


# -------------------------------------------------
# Inspección de estructura
# -------------------------------------------------
imprimir_control_geografico(
    "FUENTE TM-ESTACIONES",
    estaciones,
)

display(
    campos_relevantes(
        estaciones,
        patron_estaciones,
    )
)


print()
imprimir_control_geografico(
    "FUENTE TM-PARADEROS",
    paraderos,
)

display(
    campos_relevantes(
        paraderos,
        patron_paraderos,
    )
)


# -------------------------------------------------
# Territorialización espacial
# -------------------------------------------------
territorios_movilidad = mr_base[
    [
        "codigo_localidad",
        "localidad",
        "geometry",
    ]
].rename(
    columns={
        "codigo_localidad":
            "codigo_localidad_espacial",
        "localidad":
            "localidad_espacial",
    }
)


estaciones_geo = estaciones.to_crs(
    mr.crs
).copy()

paraderos_geo = paraderos.to_crs(
    mr.crs
).copy()


estaciones_localizadas = gpd.sjoin(
    estaciones_geo,
    territorios_movilidad,
    how="left",
    predicate="within",
)

paraderos_localizados = gpd.sjoin(
    paraderos_geo,
    territorios_movilidad,
    how="left",
    predicate="within",
)


print("\nCONTROL ESPACIAL DE MOVILIDAD")

print("\nEstaciones")
print("Filas originales:", len(estaciones))
print(
    "Filas después del cruce:",
    len(estaciones_localizadas),
)
print(
    "Asignadas:",
    int(
        estaciones_localizadas[
            "codigo_localidad_espacial"
        ].notna().sum()
    ),
)
print(
    "Sin localidad:",
    int(
        estaciones_localizadas[
            "codigo_localidad_espacial"
        ].isna().sum()
    ),
)

print("\nParaderos")
print("Filas originales:", len(paraderos))
print(
    "Filas después del cruce:",
    len(paraderos_localizados),
)
print(
    "Asignados:",
    int(
        paraderos_localizados[
            "codigo_localidad_espacial"
        ].notna().sum()
    ),
)
print(
    "Sin localidad:",
    int(
        paraderos_localizados[
            "codigo_localidad_espacial"
        ].isna().sum()
    ),
)

FUENTE TM-ESTACIONES
Filas: 153
Columnas: 26
CRS: EPSG:4326
Tipos de geometría:
Point    153
Name: count, dtype: int64
Geometrías nulas: 0
Geometrías vacías: 0
Geometrías inválidas: 0
Geometrías repetidas: 0
Duplicados exactos: 0


,campo,tipo,nulos,valores_unicos,ejemplos
0,objectid,int32,0,153,1 | 2 | 3
1,numero_estacion,str,0,153,10000 | 03000 | 06000
2,nombre_estacion,str,0,152,Portal 20 de Julio | Portal Suba | Portal El D...
3,coordenada_x_estacion,float64,0,153,997836.0595 | 998139.4242 | 995130.6531
4,coordenada_y_estacion,float64,0,153,996629.2544 | 1016655.2375 | 1009444.2454
5,ubicacion_estacion,str,0,152,Kra 5 a Cl 33 Sur | Av Suba Av C. Cali | KR 87
6,troncal_estacion,str,0,13,Cr 7-10 | Suba | Calle 26
7,numero_vagones_estacion,int32,0,5,1 | 3 | 2
8,numero_accesos_estacion,int32,0,3,0 | 2 | 1
9,biciestacion_estacion,str,0,2,1 | 0



FUENTE TM-PARADEROS
Filas: 7653
Columnas: 17
CRS: EPSG:4686
Tipos de geometría:
Point    7653
Name: count, dtype: int64
Geometrías nulas: 0
Geometrías vacías: 0
Geometrías inválidas: 0
Geometrías repetidas: 2
Duplicados exactos: 0


,campo,tipo,nulos,valores_unicos,ejemplos
0,objectid,int64,0,7653,6801 | 6802 | 6803
1,zona_parad,int64,0,15,12 | 13 | 11
2,nombre_par,str,2,3538,Santa Bárbara | Usme Rural | Usme Centro
3,direccion_,str,1,6153,Vía Pasquilla | KR 3 - CL 139 Sur | KR 3 - CL ...
4,localidad_,int64,0,20,19 | 5 | 4
5,longitud_p,float64,0,7651,-74.1466260004 | -74.1283850003 | -74.1281940004
6,latitud_pa,float64,0,7651,4.40493099968 | 4.46837299983 | 4.46841000041
7,coordenada,float64,0,7651,992328.3843 | 994353.4881 | 994374.6862
8,coordena_1,float64,0,7651,978849.5702 | 985864.8849 | 985868.975



CONTROL ESPACIAL DE MOVILIDAD

Estaciones
Filas originales: 153
Filas después del cruce: 153
Asignadas: 149
Sin localidad: 4

Paraderos
Filas originales: 7653
Filas después del cruce: 7653
Asignados: 7653
Sin localidad: 0


In [16]:
# =================================================
# ESTACIONES FUERA DE LOS POLÍGONOS
# =================================================
estaciones_fuera = estaciones_localizadas.loc[
    estaciones_localizadas[
        "codigo_localidad_espacial"
    ].isna()
].drop(
    columns=["index_right"],
    errors="ignore",
).copy()

estaciones_fuera["longitud"] = (
    estaciones_fuera.geometry.x
)
estaciones_fuera["latitud"] = (
    estaciones_fuera.geometry.y
)

estaciones_fuera_proyectadas = (
    estaciones_fuera.to_crs(epsg=3116)
)

territorios_movilidad_proyectados = (
    mr_base
    .to_crs(epsg=3116)[
        [
            "codigo_localidad",
            "localidad",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "codigo_localidad":
                "codigo_localidad_cercana",
            "localidad":
                "localidad_cercana",
        }
    )
)

estaciones_fuera_audit = gpd.sjoin_nearest(
    estaciones_fuera_proyectadas,
    territorios_movilidad_proyectados,
    how="left",
    distance_col="distancia_bogota_m",
)

print("ESTACIONES FUERA DE BOGOTÁ")
display(
    estaciones_fuera_audit[
        [
            "numero_estacion",
            "nombre_estacion",
            "ubicacion_estacion",
            "troncal_estacion",
            "codigo_localidad_cercana",
            "localidad_cercana",
            "distancia_bogota_m",
            "longitud",
            "latitud",
        ]
    ].round(
        {
            "distancia_bogota_m": 2,
            "longitud": 6,
            "latitud": 6,
        }
    )
)


# =================================================
# CONTRASTE TERRITORIAL DE PARADEROS
# =================================================
paraderos_localizados[
    "codigo_localidad_fuente"
] = (
    pd.to_numeric(
        paraderos_localizados["localidad_"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

paraderos_localizados["localidad_fuente"] = (
    paraderos_localizados[
        "codigo_localidad_fuente"
    ].map(nombres_por_codigo)
)

diferencias_paraderos = (
    paraderos_localizados[
        "codigo_localidad_espacial"
    ].notna()
    & (
        paraderos_localizados[
            "codigo_localidad_fuente"
        ]
        != paraderos_localizados[
            "codigo_localidad_espacial"
        ]
    )
)

print("\nCONTROL TERRITORIAL DE PARADEROS")
print(
    "Códigos de fuente faltantes:",
    int(
        paraderos_localizados[
            "codigo_localidad_fuente"
        ].isna().sum()
    ),
)
print(
    "Códigos no presentes en MR:",
    sorted(
        set(
            paraderos_localizados[
                "codigo_localidad_fuente"
            ].dropna()
        )
        - codigos_mr
    ),
)
print(
    "Coincidencias código-geometría:",
    int((~diferencias_paraderos).sum()),
)
print(
    "Discrepancias código-geometría:",
    int(diferencias_paraderos.sum()),
)


# =================================================
# REGLA TERRITORIAL CANÓNICA DE PARADEROS
# =================================================
codigo_fuente_valido = (
    paraderos_localizados[
        "codigo_localidad_fuente"
    ].isin(codigos_mr)
)

discrepancia_con_codigo_valido = (
    codigo_fuente_valido
    & (
        paraderos_localizados[
            "codigo_localidad_fuente"
        ]
        != paraderos_localizados[
            "codigo_localidad_espacial"
        ]
    )
)

# Se conserva el código de la fuente cuando es válido
paraderos_localizados[
    "codigo_localidad_modelo"
] = paraderos_localizados[
    "codigo_localidad_fuente"
]

paraderos_localizados[
    "regla_territorial"
] = "codigo_fuente"

# Si el código es inválido o no existe, se utiliza
# la localidad determinada por la geometría
paraderos_localizados.loc[
    ~codigo_fuente_valido,
    "codigo_localidad_modelo",
] = paraderos_localizados.loc[
    ~codigo_fuente_valido,
    "codigo_localidad_espacial",
]

paraderos_localizados.loc[
    ~codigo_fuente_valido,
    "regla_territorial",
] = "geometria_por_codigo_invalido"


# Controles de aceptación
assert int(discrepancia_con_codigo_valido.sum()) == 0
assert paraderos_localizados[
    "codigo_localidad_modelo"
].notna().all()
assert set(
    paraderos_localizados[
        "codigo_localidad_modelo"
    ]
) <= codigos_mr
assert len(paraderos_localizados) == len(paraderos)


print("\nRESULTADO TERRITORIAL DE PARADEROS")
print(
    paraderos_localizados[
        "regla_territorial"
    ].value_counts()
)

print("\nREGISTRO CON CÓDIGO DE FUENTE INVÁLIDO")
display(
    paraderos_localizados.loc[
        ~codigo_fuente_valido,
        [
            "objectid",
            "nombre_par",
            "direccion_",
            "localidad_",
            "codigo_localidad_fuente",
            "codigo_localidad_espacial",
            "localidad_espacial",
            "codigo_localidad_modelo",
            "regla_territorial",
        ],
    ]
)

ESTACIONES FUERA DE BOGOTÁ


,numero_estacion,nombre_estacion,ubicacion_estacion,troncal_estacion,codigo_localidad_cercana,localidad_cercana,distancia_bogota_m,longitud,latitud
40,07505,León XIII,Autopista Sur - CL 48,Soacha,07,BOSA,676.98,-74.193136,4.592174
44,07504,Terreros - Hospital Cardio Vascular,Autopista Sur - Av Terreros,Soacha,07,BOSA,1176.83,-74.199530,4.588976
70,07506,La Despensa,Autopista Sur - Kr 20A,Soacha,07,BOSA,289.97,-74.188119,4.594608
136,07503,San Mateo - CC Unisur,Autopista Sur - Kr 33,Soacha,07,BOSA,1493.48,-74.205464,4.585989



CONTROL TERRITORIAL DE PARADEROS
Códigos de fuente faltantes: 0
Códigos no presentes en MR: ['00']
Coincidencias código-geometría: 7652
Discrepancias código-geometría: 1

RESULTADO TERRITORIAL DE PARADEROS
regla_territorial
codigo_fuente                    7652
geometria_por_codigo_invalido       1
Name: count, dtype: int64

REGISTRO CON CÓDIGO DE FUENTE INVÁLIDO


,objectid,nombre_par,direccion_,localidad_,codigo_localidad_fuente,codigo_localidad_espacial,localidad_espacial,codigo_localidad_modelo,regla_territorial
7643,14856,NaN,NaN,0,00,11,SUBA,11,geometria_por_codigo_invalido


### Indicadores territoriales de movilidad

Se excluyen cuatro estaciones pertenecientes a la troncal Soacha porque la
unidad de análisis está restringida a las 20 localidades de Bogotá.

Las estaciones restantes se asignan mediante intersección espacial. Los
paraderos utilizan el código territorial de la fuente y, cuando este no es un
código válido de localidad, se utiliza el código obtenido mediante la geometría.

\[
\text{Estaciones por km²}
=
\frac{\text{Estaciones troncales de la localidad}}
{\text{Área de la localidad en km²}}
\]

\[
\text{Paraderos por km²}
=
\frac{\text{Paraderos zonales de la localidad}}
{\text{Área de la localidad en km²}}
\]

Ambos indicadores representan disponibilidad: valores mayores implican menor
prioridad territorial. Primero se normalizarán por separado y posteriormente
se combinarán dentro de una única dimensión de movilidad.

In [17]:
# =================================================
# ESTACIONES DE BOGOTÁ
# =================================================
estaciones_bogota = estaciones_localizadas.loc[
    estaciones_localizadas[
        "codigo_localidad_espacial"
    ].notna()
].copy()

indicador_estaciones = (
    estaciones_bogota
    .groupby(
        "codigo_localidad_espacial",
        as_index=False,
    )
    .agg(
        estaciones_troncales=(
            "numero_estacion",
            "nunique",
        )
    )
    .rename(
        columns={
            "codigo_localidad_espacial":
                "codigo_localidad"
        }
    )
)


# =================================================
# PARADEROS DE BOGOTÁ
# =================================================
indicador_paraderos = (
    paraderos_localizados
    .groupby(
        "codigo_localidad_modelo",
        as_index=False,
    )
    .agg(
        paraderos_zonales=(
            "objectid",
            "nunique",
        )
    )
    .rename(
        columns={
            "codigo_localidad_modelo":
                "codigo_localidad"
        }
    )
)


# Permite ejecutar nuevamente la celda
base_indicadores = base_indicadores.drop(
    columns=[
        "estaciones_troncales",
        "paraderos_zonales",
        "estaciones_por_km2",
        "paraderos_por_km2",
    ],
    errors="ignore",
)

base_indicadores = (
    base_indicadores
    .merge(
        indicador_estaciones,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
    .merge(
        indicador_paraderos,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
)

base_indicadores = gpd.GeoDataFrame(
    base_indicadores,
    geometry="geometry",
    crs=base_territorial.crs,
)


# Una localidad sin registros en una fuente completa
# representa una disponibilidad observada igual a cero
columnas_conteo_movilidad = [
    "estaciones_troncales",
    "paraderos_zonales",
]

base_indicadores[
    columnas_conteo_movilidad
] = (
    base_indicadores[
        columnas_conteo_movilidad
    ]
    .fillna(0)
    .astype(int)
)


# Densidades
base_indicadores["estaciones_por_km2"] = (
    base_indicadores["estaciones_troncales"]
    / base_indicadores["area_km2"]
)

base_indicadores["paraderos_por_km2"] = (
    base_indicadores["paraderos_zonales"]
    / base_indicadores["area_km2"]
)


# =================================================
# CONTROLES
# =================================================
assert len(base_indicadores) == 20
assert base_indicadores["codigo_localidad"].is_unique
assert base_indicadores["estaciones_troncales"].sum() == 149
assert base_indicadores["paraderos_zonales"].sum() == 7653
assert base_indicadores[
    [
        "estaciones_por_km2",
        "paraderos_por_km2",
    ]
].notna().all().all()

resumen_movilidad, atipicos_movilidad = auditoria_iqr(
    base_indicadores,
    [
        "estaciones_por_km2",
        "paraderos_por_km2",
    ],
)


print("INDICADORES DE MOVILIDAD")
print(
    "Estaciones originales:",
    len(estaciones),
)
print(
    "Estaciones excluidas por pertenecer a Soacha:",
    len(estaciones_fuera),
)
print(
    "Estaciones incluidas en Bogotá:",
    int(base_indicadores["estaciones_troncales"].sum()),
)
print(
    "Paraderos incluidos:",
    int(base_indicadores["paraderos_zonales"].sum()),
)

print("\nLOCALIDADES SIN ESTACIONES TRONCALES")
print(
    base_indicadores.loc[
        base_indicadores["estaciones_troncales"] == 0,
        "localidad",
    ].tolist()
)

display(
    base_indicadores[
        [
            "codigo_localidad",
            "localidad",
            "area_km2",
            "estaciones_troncales",
            "estaciones_por_km2",
            "paraderos_zonales",
            "paraderos_por_km2",
        ]
    ].round(2)
)

print("\nRESUMEN DE DISTRIBUCIONES")
display(resumen_movilidad.round(2))

print("\nVALORES ATÍPICOS DOCUMENTADOS")
display(atipicos_movilidad.round(2))

INDICADORES DE MOVILIDAD
Estaciones originales: 153
Estaciones excluidas por pertenecer a Soacha: 4
Estaciones incluidas en Bogotá: 149
Paraderos incluidos: 7653

LOCALIDADES SIN ESTACIONES TRONCALES
['SUMAPAZ']


,codigo_localidad,localidad,area_km2,estaciones_troncales,estaciones_por_km2,paraderos_zonales,paraderos_por_km2
0,01,USAQUEN,65.20,10,0.15,698,10.71
1,02,CHAPINERO,38.01,9,0.24,333,8.76
2,03,SANTA FE,45.17,14,0.31,194,4.29
3,04,SAN CRISTOBAL,49.10,3,0.06,408,8.31
4,05,USME,215.07,2,0.01,327,1.52
5,06,TUNJUELITO,9.91,4,0.40,192,19.37
6,07,BOSA,23.93,5,0.21,485,20.26
7,08,KENNEDY,38.59,10,0.26,926,24.00
8,09,FONTIBON,33.28,2,0.06,388,11.66
9,10,ENGATIVA,35.88,12,0.33,784,21.85



RESUMEN DE DISTRIBUCIONES


,indicador,minimo,q1,mediana,q3,maximo,limite_inferior,limite_superior,cantidad_atipicos,asimetria
0,estaciones_por_km2,0.0,0.11,0.28,0.60,1.38,-0.61,1.32,1,1.02
1,paraderos_por_km2,0.0,8.38,18.41,20.64,25.64,-10.02,39.04,0,-0.41



VALORES ATÍPICOS DOCUMENTADOS


,indicador,localidad,valor,tipo
0,estaciones_por_km2,LOS MARTIRES,1.38,superior


### Auditoría semántica de ambiente e infraestructura

La fuente ambiental se revisa para identificar la unidad de cada conflicto,
su cobertura territorial y la existencia de duplicados.

La fuente de parques se revisa para determinar si contiene área física. En
ausencia de esa variable, el conteo de parques puede conservarse como proxy
exploratorio, pero no debe interpretarse como área de parques por habitante.

In [18]:
import unicodedata


parques = leer_archivo_modelo(
    rutas_unicas["INFRA-PARQUES"]
)

ambiente = leer_archivo_modelo(
    rutas_unicas["AMB-SAC"]
)


def normalizar_nombre_territorial(valor):
    if pd.isna(valor):
        return pd.NA

    texto = unicodedata.normalize(
        "NFKD",
        str(valor).strip().upper(),
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    return " ".join(texto.split())


def duplicados_exactos_tabulares(df):
    return int(
        df.astype("string").duplicated().sum()
    )


# =================================================
# INFRAESTRUCTURA: PARQUES
# =================================================
print("FUENTE INFRA-PARQUES")
print("Filas:", len(parques))
print("Columnas:", len(parques.columns))
print("Columnas disponibles:", parques.columns.tolist())
print(
    "Duplicados exactos:",
    duplicados_exactos_tabulares(parques),
)
print(
    "Códigos de parque únicos:",
    parques["Codigo Parque"].nunique(),
)

display(ficha_campos(parques))


# Catálogo territorial normalizado
catalogo_localidades = base_territorial[
    [
        "codigo_localidad",
        "localidad",
    ]
].copy()

catalogo_localidades[
    "localidad_normalizada"
] = catalogo_localidades["localidad"].map(
    normalizar_nombre_territorial
)

parques_audit = parques.copy()

parques_audit[
    "localidad_normalizada"
] = parques_audit["Nombre Localidad"].map(
    normalizar_nombre_territorial
)

parques_audit = parques_audit.merge(
    catalogo_localidades,
    on="localidad_normalizada",
    how="left",
    validate="many_to_one",
)

print("\nCONTROL TERRITORIAL DE PARQUES")
print(
    "Localidades informadas por la fuente:",
    parques["Nombre Localidad"].nunique(),
)
print(
    "Registros sin correspondencia territorial:",
    int(
        parques_audit[
            "codigo_localidad"
        ].isna().sum()
    ),
)

print("Nombres sin correspondencia:")
print(
    parques_audit.loc[
        parques_audit["codigo_localidad"].isna(),
        "Nombre Localidad",
    ].drop_duplicates().tolist()
)

print("\nPARQUES POR LOCALIDAD")
display(
    parques_audit
    .groupby(
        [
            "codigo_localidad",
            "localidad",
        ],
        dropna=False,
    )
    .agg(
        parques_registrados=(
            "Codigo Parque",
            "nunique",
        )
    )
    .reset_index()
)


# =================================================
# AMBIENTE
# =================================================
print("\nFUENTE AMB-SAC")
print("Filas:", len(ambiente))
print("Columnas:", len(ambiente.columns))
print(
    "Duplicados exactos:",
    duplicados_exactos_tabulares(ambiente),
)

patron_ambiente = (
    r"amb|conf|local|cod|^id$|object|fecha|ano|año|"
    r"tipo|situacion|categoria|estado|descripcion"
)

display(
    campos_relevantes(
        ambiente,
        patron_ambiente,
    )
)


# Control preliminar del código territorial ambiental
ambiente_audit = ambiente.copy()

ambiente_audit[
    "codigo_localidad_modelo"
] = (
    pd.to_numeric(
        ambiente_audit["cod_locali"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

print("\nCONTROL TERRITORIAL DE AMBIENTE")
print(
    "Códigos faltantes:",
    int(
        ambiente_audit[
            "codigo_localidad_modelo"
        ].isna().sum()
    ),
)
print(
    "Códigos no presentes en MR:",
    sorted(
        set(
            ambiente_audit[
                "codigo_localidad_modelo"
            ].dropna()
        )
        - codigos_mr
    ),
)
print(
    "Localidades representadas:",
    ambiente_audit[
        "codigo_localidad_modelo"
    ].nunique(),
)

FUENTE INFRA-PARQUES
Filas: 139
Columnas: 5
Columnas disponibles: ['Codigo Parque', 'Nombre Parque', 'Nombre Localidad', 'Tipologia', 'Administracion']
Duplicados exactos: 0
Códigos de parque únicos: 135


,campo,tipo,nulos,valores_unicos,ejemplos
0,Codigo Parque,str,0,135,01-012 | 01-023 | 01-064
1,Nombre Parque,str,0,138,La Vida | Servita | Nueva Autopista
2,Nombre Localidad,str,0,19,USAQUÉN | CHAPINERO | SANTA FE
3,Tipologia,str,0,1,ESTRUCTURANTE
4,Administracion,str,0,1,IDRD



CONTROL TERRITORIAL DE PARQUES
Localidades informadas por la fuente: 19
Registros sin correspondencia territorial: 6
Nombres sin correspondencia:
['MÁRTIRES', 'LA CANDELARIA']

PARQUES POR LOCALIDAD


,codigo_localidad,localidad,parques_registrados
0,01,USAQUEN,5
1,02,CHAPINERO,3
2,03,SANTA FE,8
3,04,SAN CRISTOBAL,6
4,05,USME,7
5,06,TUNJUELITO,2
6,07,BOSA,11
7,08,KENNEDY,13
8,09,FONTIBON,5
9,10,ENGATIVA,13



FUENTE AMB-SAC
Filas: 1313
Columnas: 14
Duplicados exactos: 24


,campo,tipo,nulos,valores_unicos,ejemplos
0,categoria,str,2,61,Vertimientos aguas residuales domesticas | Dis...
1,localidad,str,1,27,USAQUÉN | CHAPINERO | SANTA FE
2,cod_locali,str,26,48,1 | 2 | 3
3,codigo_sac,str,56,47,EH003 | CR002 | EH001



CONTROL TERRITORIAL DE AMBIENTE
Códigos faltantes: 56
Códigos no presentes en MR: []
Localidades representadas: 20


In [19]:
# =================================================
# CORRECCIÓN REPRODUCIBLE DE ALIAS DE PARQUES
# =================================================
alias_localidades_parques = {
    "MARTIRES": "LOS MARTIRES",
    "LA CANDELARIA": "CANDELARIA",
}

parques_corregidos = parques.copy()

parques_corregidos[
    "localidad_normalizada"
] = (
    parques_corregidos["Nombre Localidad"]
    .map(normalizar_nombre_territorial)
    .replace(alias_localidades_parques)
)

parques_corregidos = parques_corregidos.merge(
    catalogo_localidades,
    on="localidad_normalizada",
    how="left",
    validate="many_to_one",
)

print("CONTROL CORREGIDO DE PARQUES")
print("Registros:", len(parques_corregidos))
print(
    "Códigos de parque únicos:",
    parques_corregidos["Codigo Parque"].nunique(),
)
print(
    "Registros sin localidad:",
    int(
        parques_corregidos[
            "codigo_localidad"
        ].isna().sum()
    ),
)

resumen_parques = (
    parques_corregidos
    .groupby(
        [
            "codigo_localidad",
            "localidad",
        ],
        as_index=False,
    )
    .agg(
        parques_registrados=(
            "Codigo Parque",
            "nunique",
        )
    )
)

resumen_parques_completo = (
    base_territorial[
        [
            "codigo_localidad",
            "localidad",
        ]
    ]
    .merge(
        resumen_parques,
        on=[
            "codigo_localidad",
            "localidad",
        ],
        how="left",
        validate="one_to_one",
    )
)

resumen_parques_completo[
    "parques_registrados"
] = (
    resumen_parques_completo[
        "parques_registrados"
    ]
    .fillna(0)
    .astype(int)
)

print(
    "Localidades sin parques registrados:",
    resumen_parques_completo.loc[
        resumen_parques_completo[
            "parques_registrados"
        ] == 0,
        "localidad",
    ].tolist(),
)

display(resumen_parques_completo)


# =================================================
# DEPURACIÓN Y TERRITORIALIZACIÓN AMBIENTAL
# =================================================
print("\nCOLUMNAS COMPLETAS DE AMBIENTE")
print(ambiente.columns.tolist())

ambiente_sin_duplicados = (
    ambiente.drop_duplicates().copy()
)

ambiente_sin_duplicados[
    "codigo_numerico"
] = pd.to_numeric(
    ambiente_sin_duplicados["cod_locali"],
    errors="coerce",
)

ambiente_sin_duplicados[
    "codigo_desde_campo"
] = (
    ambiente_sin_duplicados[
        "codigo_numerico"
    ]
    .where(
        ambiente_sin_duplicados[
            "codigo_numerico"
        ].between(1, 20)
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)


# Recuperación mediante el nombre de localidad
alias_localidades_ambiente = {
    "MARTIRES": "LOS MARTIRES",
    "LA CANDELARIA": "CANDELARIA",
    "SANTAFE": "SANTA FE",
    "CIUDAD BOLIVAR": "CIUDAD BOLIVAR",
}

ambiente_sin_duplicados[
    "localidad_normalizada"
] = (
    ambiente_sin_duplicados["localidad"]
    .map(normalizar_nombre_territorial)
    .replace(alias_localidades_ambiente)
)

codigo_por_nombre = (
    catalogo_localidades
    .set_index("localidad_normalizada")[
        "codigo_localidad"
    ]
    .to_dict()
)

ambiente_sin_duplicados[
    "codigo_desde_nombre"
] = (
    ambiente_sin_duplicados[
        "localidad_normalizada"
    ].map(codigo_por_nombre)
)

ambiente_sin_duplicados[
    "codigo_localidad_modelo"
] = (
    ambiente_sin_duplicados[
        "codigo_desde_campo"
    ].fillna(
        ambiente_sin_duplicados[
            "codigo_desde_nombre"
        ]
    )
)


codigo_campo_valido = ambiente_sin_duplicados[
    "codigo_desde_campo"
].notna()

recuperado_por_nombre = (
    ambiente_sin_duplicados[
        "codigo_desde_campo"
    ].isna()
    & ambiente_sin_duplicados[
        "codigo_desde_nombre"
    ].notna()
)

sin_localidad_ambiente = ambiente_sin_duplicados[
    "codigo_localidad_modelo"
].isna()


print("\nCONTROL DEPURADO DE AMBIENTE")
print("Filas originales:", len(ambiente))
print(
    "Duplicados exactos eliminados:",
    len(ambiente) - len(ambiente_sin_duplicados),
)
print(
    "Filas después de eliminar duplicados:",
    len(ambiente_sin_duplicados),
)
print(
    "Códigos territoriales válidos directamente:",
    int(codigo_campo_valido.sum()),
)
print(
    "Códigos recuperados por nombre:",
    int(recuperado_por_nombre.sum()),
)
print(
    "Filas aún sin localidad:",
    int(sin_localidad_ambiente.sum()),
)
print(
    "Filas sin codigo_sac:",
    int(
        ambiente_sin_duplicados[
            "codigo_sac"
        ].isna().sum()
    ),
)
print(
    "Códigos SAC únicos:",
    ambiente_sin_duplicados[
        "codigo_sac"
    ].nunique(),
)


print("\nVALORES TERRITORIALES NO NUMÉRICOS O FALTANTES")
display(
    ambiente_sin_duplicados.loc[
        ambiente_sin_duplicados[
            "codigo_desde_campo"
        ].isna(),
        [
            "localidad",
            "cod_locali",
            "codigo_sac",
            "categoria",
            "codigo_desde_nombre",
            "codigo_localidad_modelo",
        ],
    ]
    .drop_duplicates()
    .head(30)
)


print("\nFILAS AÚN SIN LOCALIDAD")
display(
    ambiente_sin_duplicados.loc[
        sin_localidad_ambiente,
        [
            "localidad",
            "cod_locali",
            "codigo_sac",
            "categoria",
        ],
    ]
    .drop_duplicates()
)


# Granularidad de los códigos de conflicto
resumen_codigo_sac = (
    ambiente_sin_duplicados.loc[
        ambiente_sin_duplicados[
            "codigo_sac"
        ].notna()
    ]
    .groupby(
        "codigo_sac",
        as_index=False,
    )
    .agg(
        filas=("codigo_sac", "size"),
        categorias=("categoria", "nunique"),
        localidades=(
            "codigo_localidad_modelo",
            "nunique",
        ),
    )
)

print("\nDISTRIBUCIÓN DE FILAS POR CODIGO_SAC")
display(
    resumen_codigo_sac["filas"]
    .describe()
    .to_frame()
    .T
    .round(2)
)

print("\nCÓDIGOS SAC CON MÁS FILAS")
display(
    resumen_codigo_sac
    .sort_values("filas", ascending=False)
    .head(15)
)

CONTROL CORREGIDO DE PARQUES
Registros: 139
Códigos de parque únicos: 135
Registros sin localidad: 0
Localidades sin parques registrados: ['SUMAPAZ']


,codigo_localidad,localidad,parques_registrados
0,01,USAQUEN,5
1,02,CHAPINERO,3
2,03,SANTA FE,8
3,04,SAN CRISTOBAL,6
4,05,USME,7
5,06,TUNJUELITO,2
6,07,BOSA,11
7,08,KENNEDY,13
8,09,FONTIBON,5
9,10,ENGATIVA,13



COLUMNAS COMPLETAS DE AMBIENTE
['latitude', 'longitude', 'altitude', 'geometry', 'categoria', 'localidad', 'direccion', 'cod_locali', 'grupo_sac', 'sac', 'codigo_sac', 'gestor', 'cord_x', 'cord_y']

CONTROL DEPURADO DE AMBIENTE
Filas originales: 1313
Duplicados exactos eliminados: 24
Filas después de eliminar duplicados: 1289
Códigos territoriales válidos directamente: 1233
Códigos recuperados por nombre: 25
Filas aún sin localidad: 31
Filas sin codigo_sac: 56
Códigos SAC únicos: 47

VALORES TERRITORIALES NO NUMÉRICOS O FALTANTES


,localidad,cod_locali,codigo_sac,categoria,codigo_desde_nombre,codigo_localidad_modelo
486,KENNEDY,NaN,NaN,Disposición inadecuada de residuos sólidos,08,08
487,ADRIÁN HERNÁNDEZ,101.032.876.284,NaN,CR004,NaN,<NA>
524,KENNEDY,NaN,NaN,Deficiencias en mantenimiento de arbolado urbano,08,08
525,ADRIÁN HERNÁNDEZ,103.600.985.453,NaN,ES003,NaN,<NA>
526,KENNEDY,NaN,NaN,Vectores,08,08
527,ADRIÁN HERNÁNDEZ,105.253.985.933,NaN,FV001,NaN,<NA>
528,KENNEDY,NaN,NaN,Zonas con riesgo de incendio forestal,08,08
529,ADRIÁN HERNÁNDEZ,105.369.940.682,NaN,GI001,NaN,<NA>
530,KENNEDY,NaN,NaN,Necesidad de procesos de educación ambiental c...,08,08
531,ADRIÁN HERNÁNDEZ,100.812.116.252,NaN,PE002,NaN,<NA>



FILAS AÚN SIN LOCALIDAD


,localidad,cod_locali,codigo_sac,categoria
487,ADRIÁN HERNÁNDEZ,101.032.876.284,NaN,CR004
525,ADRIÁN HERNÁNDEZ,103.600.985.453,NaN,ES003
527,ADRIÁN HERNÁNDEZ,105.253.985.933,NaN,FV001
529,ADRIÁN HERNÁNDEZ,105.369.940.682,NaN,GI001
531,ADRIÁN HERNÁNDEZ,100.812.116.252,NaN,PE002
533,ADRIÁN HERNÁNDEZ,101.152.805.076,NaN,EI003
536,ADRIÁN HERNÁNDEZ,1.013.286.984,NaN,ES003
538,ADRIÁN HERNÁNDEZ,102.822.155.033,NaN,CR003
540,ADRIÁN HERNÁNDEZ,102.702.674.191,NaN,CR003
542,ADRIÁN HERNÁNDEZ,102.678.995.845,NaN,CR003



DISTRIBUCIÓN DE FILAS POR CODIGO_SAC


,count,mean,std,min,25%,50%,75%,max
filas,47.0,26.23,67.71,1.0,3.5,10.0,22.5,461.0



CÓDIGOS SAC CON MÁS FILAS


,codigo_sac,filas,categorias,localidades
9,CR004,461,2,20
24,FD001,87,1,11
37,GM002,78,1,5
7,CR002,64,2,14
43,PE002,50,2,4
14,EH001,46,1,11
28,FV001,35,1,8
12,EA003,29,1,1
19,EI003,29,1,11
23,ES004,29,1,5


### Indicadores de ambiente e infraestructura

Para ambiente se consideran únicamente registros no duplicados que poseen
código SAC y localidad válida. Cada fila representa un registro ambiental
clasificado; `codigo_sac` se conserva adicionalmente para medir la diversidad
de tipos de conflicto.

\[
\text{Conflictos ambientales registrados por km²}
=
\frac{\text{Registros ambientales clasificados}}
{\text{Área de la localidad en km²}}
\]

La fuente de parques no contiene superficie. Por ello se construye el proxy:

\[
\text{Parques registrados por 10.000 habitantes}
=
\frac{\text{Parques únicos registrados}}
{\text{Población total}} \times 10.000
\]

Este proxy no equivale a área de parque, calidad, accesibilidad ni capacidad.
Su efecto será evaluado mediante un escenario de sensibilidad que lo excluya.

In [20]:
# =================================================
# UNIVERSO ANALÍTICO AMBIENTAL
# =================================================
ambiente_valido = ambiente_sin_duplicados.loc[
    ambiente_sin_duplicados[
        "codigo_localidad_modelo"
    ].isin(codigos_mr)
    & ambiente_sin_duplicados[
        "codigo_sac"
    ].notna()
].copy()


indicador_ambiente = (
    ambiente_valido
    .groupby(
        "codigo_localidad_modelo",
        as_index=False,
    )
    .agg(
        conflictos_ambientales_registrados=(
            "codigo_sac",
            "size",
        ),
        tipos_conflicto_ambiental=(
            "codigo_sac",
            "nunique",
        ),
    )
    .rename(
        columns={
            "codigo_localidad_modelo":
                "codigo_localidad"
        }
    )
)


# =================================================
# PROXY DE INFRAESTRUCTURA
# =================================================
indicador_parques = (
    resumen_parques_completo[
        [
            "codigo_localidad",
            "parques_registrados",
        ]
    ].copy()
)


# Permite volver a ejecutar la celda
base_indicadores = base_indicadores.drop(
    columns=[
        "conflictos_ambientales_registrados",
        "tipos_conflicto_ambiental",
        "conflictos_ambientales_por_km2",
        "parques_registrados",
        "parques_por_10000_hab_proxy",
    ],
    errors="ignore",
)

base_indicadores = (
    base_indicadores
    .merge(
        indicador_ambiente,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
    .merge(
        indicador_parques,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
)

base_indicadores = gpd.GeoDataFrame(
    base_indicadores,
    geometry="geometry",
    crs=base_territorial.crs,
)


# Las fuentes cubren explícitamente las 20 localidades.
# Ausencia de registros equivale a conteo observado igual a cero.
columnas_conteo_ambiente_infra = [
    "conflictos_ambientales_registrados",
    "tipos_conflicto_ambiental",
    "parques_registrados",
]

base_indicadores[
    columnas_conteo_ambiente_infra
] = (
    base_indicadores[
        columnas_conteo_ambiente_infra
    ]
    .fillna(0)
    .astype(int)
)


# Indicadores territoriales
base_indicadores[
    "conflictos_ambientales_por_km2"
] = (
    base_indicadores[
        "conflictos_ambientales_registrados"
    ]
    / base_indicadores["area_km2"]
)

base_indicadores[
    "parques_por_10000_hab_proxy"
] = (
    base_indicadores["parques_registrados"]
    / base_indicadores["poblacion_2025"]
    * 10_000
)


# =================================================
# CONTROLES
# =================================================
assert len(base_indicadores) == 20
assert base_indicadores["codigo_localidad"].is_unique
assert len(ambiente_valido) == 1233
assert (
    base_indicadores[
        "conflictos_ambientales_registrados"
    ].sum()
    == len(ambiente_valido)
)
assert (
    base_indicadores[
        "parques_registrados"
    ].sum()
    == parques["Codigo Parque"].nunique()
)
assert base_indicadores[
    [
        "conflictos_ambientales_por_km2",
        "parques_por_10000_hab_proxy",
    ]
].notna().all().all()


resumen_ambiente_infra, atipicos_ambiente_infra = (
    auditoria_iqr(
        base_indicadores,
        [
            "conflictos_ambientales_por_km2",
            "parques_por_10000_hab_proxy",
        ],
    )
)


print("INDICADORES DE AMBIENTE E INFRAESTRUCTURA")
print("Registros ambientales originales:", len(ambiente))
print(
    "Duplicados ambientales eliminados:",
    len(ambiente) - len(ambiente_sin_duplicados),
)
print(
    "Registros sin identificación completa excluidos:",
    len(ambiente_sin_duplicados) - len(ambiente_valido),
)
print(
    "Registros ambientales incluidos:",
    int(
        base_indicadores[
            "conflictos_ambientales_registrados"
        ].sum()
    ),
)
print(
    "Parques únicos incluidos:",
    int(base_indicadores["parques_registrados"].sum()),
)

display(
    base_indicadores[
        [
            "codigo_localidad",
            "localidad",
            "area_km2",
            "conflictos_ambientales_registrados",
            "tipos_conflicto_ambiental",
            "conflictos_ambientales_por_km2",
            "parques_registrados",
            "parques_por_10000_hab_proxy",
        ]
    ].round(2)
)

print("\nRESUMEN DE DISTRIBUCIONES")
display(resumen_ambiente_infra.round(2))

print("\nVALORES ATÍPICOS DOCUMENTADOS")
display(atipicos_ambiente_infra.round(2))

INDICADORES DE AMBIENTE E INFRAESTRUCTURA
Registros ambientales originales: 1313
Duplicados ambientales eliminados: 24
Registros sin identificación completa excluidos: 56
Registros ambientales incluidos: 1233
Parques únicos incluidos: 135


,codigo_localidad,localidad,area_km2,conflictos_ambientales_registrados,tipos_conflicto_ambiental,conflictos_ambientales_por_km2,parques_registrados,parques_por_10000_hab_proxy
0,01,USAQUEN,65.20,71,17,1.09,5,0.09
1,02,CHAPINERO,38.01,99,20,2.60,3,0.19
2,03,SANTA FE,45.17,99,15,2.19,8,0.71
3,04,SAN CRISTOBAL,49.10,16,6,0.33,6,0.15
4,05,USME,215.07,116,24,0.54,7,0.18
5,06,TUNJUELITO,9.91,71,19,7.16,2,0.11
6,07,BOSA,23.93,50,12,2.09,11,0.14
7,08,KENNEDY,38.59,79,3,2.05,13,0.12
8,09,FONTIBON,33.28,14,1,0.42,5,0.13
9,10,ENGATIVA,35.88,38,9,1.06,13,0.16



RESUMEN DE DISTRIBUCIONES


,indicador,minimo,q1,mediana,q3,maximo,limite_inferior,limite_superior,cantidad_atipicos,asimetria
0,conflictos_ambientales_por_km2,0.08,0.51,1.57,3.66,18.65,-4.22,8.39,3,1.86
1,parques_por_10000_hab_proxy,0.00,0.13,0.18,0.29,1.21,-0.12,0.54,3,2.01



VALORES ATÍPICOS DOCUMENTADOS


,indicador,localidad,valor,tipo
0,conflictos_ambientales_por_km2,ANTONIO NARIÑO,16.39,superior
1,conflictos_ambientales_por_km2,CANDELARIA,14.08,superior
2,conflictos_ambientales_por_km2,RAFAEL URIBE URIBE,18.65,superior
3,parques_por_10000_hab_proxy,SANTA FE,0.71,superior
4,parques_por_10000_hab_proxy,BARRIOS UNIDOS,0.74,superior
5,parques_por_10000_hab_proxy,CANDELARIA,1.21,superior


### Auditoría de vulnerabilidad económica y seguridad

La fuente RIVI se revisa por archivo y periodo. La tasa de vendedores informales
debe utilizar población correspondiente al mismo año y conservar explícitamente
su carácter histórico.

La fuente de cuadrantes policiales se revisa para identificar la llave única de
cada cuadrante y el código territorial, evitando contar filas duplicadas o
identificadores técnicos.

In [21]:
# =================================================
# VULNERABILIDAD ECONÓMICA: FIN-RIVI
# =================================================
rivi_archivos_leidos = {}
resumen_archivos_rivi = []

for ruta in rutas_rivi:
    datos = leer_archivo_modelo(ruta)
    rivi_archivos_leidos[ruta.name] = datos

    anios_en_nombre = re.findall(
        r"20\d{2}",
        ruta.name,
    )

    resumen_archivos_rivi.append(
        {
            "archivo": ruta.name,
            "anio_en_nombre": (
                anios_en_nombre[0]
                if anios_en_nombre
                else None
            ),
            "filas": len(datos),
            "columnas": len(datos.columns),
            "localidades": (
                datos["NumeroLocalidad"].nunique()
                if "NumeroLocalidad" in datos.columns
                else None
            ),
            "duplicados_codigo": (
                int(
                    datos.duplicated(
                        subset=["NumeroLocalidad"]
                    ).sum()
                )
                if "NumeroLocalidad" in datos.columns
                else None
            ),
            "nulos_numero": (
                int(datos["Numero"].isna().sum())
                if "Numero" in datos.columns
                else None
            ),
            "duplicados_exactos": (
                duplicados_exactos_tabulares(datos)
            ),
        }
    )

resumen_archivos_rivi = pd.DataFrame(
    resumen_archivos_rivi
)

print("ARCHIVOS FIN-RIVI")
display(resumen_archivos_rivi)


rivi_consolidado = pd.concat(
    [
        datos.assign(archivo=nombre)
        for nombre, datos
        in rivi_archivos_leidos.items()
    ],
    ignore_index=True,
)

print("\nFIN-RIVI CONSOLIDADO")
print("Filas:", len(rivi_consolidado))
print("Columnas:", rivi_consolidado.columns.tolist())
print(
    "Duplicados archivo-localidad:",
    int(
        rivi_consolidado.duplicated(
            subset=[
                "archivo",
                "NumeroLocalidad",
            ]
        ).sum()
    ),
)

display(ficha_campos(rivi_consolidado))


print("\nRESUMEN NUMÉRICO POR ARCHIVO")
display(
    rivi_consolidado
    .groupby("archivo")
    .agg(
        filas=("NumeroLocalidad", "size"),
        localidades=(
            "NumeroLocalidad",
            "nunique",
        ),
        numero_min=("Numero", "min"),
        numero_mediana=("Numero", "median"),
        numero_max=("Numero", "max"),
        numero_total=("Numero", "sum"),
    )
    .reset_index()
)


# =================================================
# SEGURIDAD: CUADRANTES
# =================================================
cuadrantes = leer_archivo_modelo(
    rutas_unicas["SEG-CUADRANTES"]
)

print("\nFUENTE SEG-CUADRANTES")
print("Filas:", len(cuadrantes))
print("Columnas:", len(cuadrantes.columns))
print(
    "Duplicados exactos:",
    duplicados_exactos_tabulares(cuadrantes),
)
print("Columnas completas:")
print(cuadrantes.columns.tolist())


patron_seguridad = (
    r"cuad|local|cod|nombre|nom|^id$|object|"
    r"unidad|estacion|fecha|vigencia|turno|sector"
)

display(
    campos_relevantes(
        cuadrantes,
        patron_seguridad,
    )
)

ARCHIVOS FIN-RIVI


,archivo,anio_en_nombre,filas,columnas,localidades,duplicados_codigo,nulos_numero,duplicados_exactos
0,rivi-numero-vendedores-informales-localidad-20...,2017,21,5,21,0,0,0
1,rivi-numero-vendedores-informales-localidad-20...,2017,21,5,21,0,0,0
2,rivi-numero-vendedores-informales-localidad-20...,2018,21,5,21,0,0,0
3,rivi-numero-vendedores-informales-localidad-20...,2018,21,4,21,0,0,0
4,rivi-numero-vendedores-informales-localidad-20...,2019,21,4,21,0,0,0
5,rivi-numero-vendedores-informales-localidad-20...,2019,21,4,21,0,0,0



FIN-RIVI CONSOLIDADO
Filas: 126
Columnas: ['Id_', 'IndiceRespuesta', 'NumeroLocalidad', 'Numero', 'Porcentaje', 'archivo', 'NombreLocalidad']
Duplicados archivo-localidad: 0


,campo,tipo,nulos,valores_unicos,ejemplos
0,Id_,float64,63,21,1.0 | 2.0 | 3.0
1,IndiceRespuesta,float64,63,21,1.0 | 2.0 | 3.0
2,NumeroLocalidad,object,0,42,Usaquén | Chapinero | Santa fé
3,Numero,int64,0,105,605 | 2499 | 9867
4,Porcentaje,float64,0,125,0.01197 | 0.049445 | 0.195228
5,archivo,str,0,6,rivi-numero-vendedores-informales-localidad-20...
6,NombreLocalidad,str,63,21,Usaquén | Chapinero | Santa fé



RESUMEN NUMÉRICO POR ARCHIVO


,archivo,filas,localidades,numero_min,numero_mediana,numero_max,numero_total
0,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1947.0,9867,50541
1,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1961.0,10129,51525
2,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1992.0,10131,51823
3,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1994.0,10201,52908
4,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1979.0,10095,52919
5,rivi-numero-vendedores-informales-localidad-20...,21,21,19,1980.0,10193,53553



FUENTE SEG-CUADRANTES
Filas: 599
Columnas: 769
Duplicados exactos: 0
Columnas completas:
['type', 'properties/PCUNCUADRA', 'properties/PCUCOSEC', 'properties/PCUNOMEST', 'properties/PCUNOMCAI', 'properties/PCUCODIGO', 'properties/PCUTELEFON', 'properties/PCUCOD_ENT', 'properties/PCUDESCRIP', 'properties/PCUIULOCAL', 'properties/PCUFECHA_C', 'properties/PCUIEPOLIC', 'properties/PCUIUUPLOC', 'properties/PCUIUUPLAN', 'properties/PCUIUSCATA', 'properties/PCUIUPCUAD', 'properties/Shape_Leng', 'properties/Shape_Area', 'geometry/type', 'geometry/coordinates/0/0/0', 'geometry/coordinates/0/0/1', 'geometry/coordinates/0/1/0', 'geometry/coordinates/0/1/1', 'geometry/coordinates/0/2/0', 'geometry/coordinates/0/2/1', 'geometry/coordinates/0/3/0', 'geometry/coordinates/0/3/1', 'geometry/coordinates/0/4/0', 'geometry/coordinates/0/4/1', 'geometry/coordinates/0/5/0', 'geometry/coordinates/0/5/1', 'geometry/coordinates/0/6/0', 'geometry/coordinates/0/6/1', 'geometry/coordinates/0/7/0', 'geometry/coor

,campo,tipo,nulos,valores_unicos,ejemplos
0,properties/PCUNCUADRA,str,0,599,MEBOGMNVCCC02E19C08000033 | MEBOGMNVCCC02E19C0...
1,properties/PCUNOMEST,str,0,19,CIUDAD BOLIVAR | ANTONIO NARIÑO | PUENTE ARANDA
2,properties/PCUNOMCAI,str,0,153,SANTO DOMINGO | ARBORIZADORA ALTA | CANDELARIA
3,properties/PCUCODIGO,str,0,599,E19C08033 | E19C08036 | E19C08034
4,properties/PCUTELEFON,int64,0,573,3008089250 | 3008010150 | 3002012932
5,properties/PCUCOD_ENT,int64,0,1,137
6,properties/PCUIULOCAL,int64,0,19,19 | 15 | 16
7,properties/PCUFECHA_C,str,0,1,30/06/2026
8,properties/PCUIUSCATA,int64,0,503,2441 | 2440 | 2442
9,properties/PCUIUPCUAD,str,0,153,E19C08 | E19C01 | E19C03


### Indicadores de vulnerabilidad económica y seguridad

RIVI contiene seis observaciones semestrales entre 2017 y 2019. Para cada
periodo se calcula la tasa utilizando la población proyectada del mismo año.
El indicador territorial corresponde al promedio de las seis tasas.

Cada archivo contiene las 20 localidades y una categoría adicional denominada
`metropolitana o no definida`. Esta categoría se excluye porque no puede
asignarse territorialmente a una localidad específica.

\[
\text{Vendedores informales por 10.000 habitantes}
=
\operatorname{promedio}_{2017-2019}
\left(
\frac{\text{Vendedores del periodo}}
{\text{Población del año}} \times 10.000
\right)
\]

Este indicador representa vulnerabilidad económica histórica y será sometido
a sensibilidad por su desfase temporal.

Los cuadrantes están vigentes al 30 de junio de 2026, por lo cual se utiliza
la población proyectada de 2026.

\[
\text{Cuadrantes por 10.000 habitantes}
=
\frac{\text{Cuadrantes únicos}}
{\text{Población 2026}} \times 10.000
\]

Una tasa RIVI mayor representa mayor prioridad. Una disponibilidad mayor de
cuadrantes representa menor prioridad.

In [22]:
# =================================================
# NORMALIZACIÓN DE LOS SEIS ARCHIVOS RIVI
# =================================================
registros_rivi_normalizados = []

alias_localidades_rivi = {
    "LA CANDELARIA": "CANDELARIA",
    "MARTIRES": "LOS MARTIRES",
    "SANTAFE": "SANTA FE",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
}

for archivo, datos in rivi_archivos_leidos.items():
    coincidencia_anio = re.search(
        r"20\d{2}",
        archivo,
    )

    if coincidencia_anio is None:
        raise ValueError(
            f"No se pudo identificar el año de {archivo}"
        )

    anio = int(coincidencia_anio.group())

    # Los archivos tienen dos esquemas diferentes
    if (
        "NombreLocalidad" in datos.columns
        and datos["NombreLocalidad"].notna().any()
    ):
        nombre_localidad = datos["NombreLocalidad"]
    else:
        nombre_localidad = datos["NumeroLocalidad"]

    temporal = pd.DataFrame(
        {
            "archivo": archivo,
            "anio": anio,
            "nombre_localidad_fuente":
                nombre_localidad,
            "vendedores_informales":
                pd.to_numeric(
                    datos["Numero"],
                    errors="raise",
                ),
            "porcentaje_fuente":
                pd.to_numeric(
                    datos["Porcentaje"],
                    errors="coerce",
                ),
        }
    )

    temporal["localidad_normalizada"] = (
        temporal["nombre_localidad_fuente"]
        .map(normalizar_nombre_territorial)
        .replace(alias_localidades_rivi)
    )

    registros_rivi_normalizados.append(temporal)


rivi_normalizado = pd.concat(
    registros_rivi_normalizados,
    ignore_index=True,
)

rivi_normalizado = rivi_normalizado.merge(
    catalogo_localidades,
    on="localidad_normalizada",
    how="left",
    validate="many_to_one",
)


print("CONTROL DE HOMOLOGACIÓN RIVI")
display(
    rivi_normalizado
    .groupby("archivo")
    .agg(
        filas=("archivo", "size"),
        localidades_mapeadas=(
            "codigo_localidad",
            lambda serie: int(serie.notna().sum()),
        ),
        registros_no_mapeados=(
            "codigo_localidad",
            lambda serie: int(serie.isna().sum()),
        ),
    )
    .reset_index()
)

print("Nombres no mapeados:")
print(
    rivi_normalizado.loc[
        rivi_normalizado[
            "codigo_localidad"
        ].isna(),
        "nombre_localidad_fuente",
    ].drop_duplicates().tolist()
)


# Se excluye únicamente el agregado Bogotá
rivi_localidades = rivi_normalizado.loc[
    rivi_normalizado[
        "codigo_localidad"
    ].notna()
].copy()


# =================================================
# POBLACIÓN 2017-2019
# =================================================
poblacion_rivi = poblacion.loc[
    poblacion["ANO"].isin([2017, 2018, 2019])
    & (poblacion["CODIGO_LOCALIDAD"] != 0)
].copy()

poblacion_rivi["codigo_localidad"] = (
    poblacion_rivi["CODIGO_LOCALIDAD"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

poblacion_rivi_anual = (
    poblacion_rivi
    .groupby(
        [
            "ANO",
            "codigo_localidad",
        ],
        as_index=False,
    )
    .agg(
        poblacion_anual=("POBLACION", "sum")
    )
    .rename(columns={"ANO": "anio"})
)


rivi_periodos = rivi_localidades.merge(
    poblacion_rivi_anual,
    on=[
        "anio",
        "codigo_localidad",
    ],
    how="left",
    validate="many_to_one",
)

rivi_periodos[
    "rivi_por_10000_hab_periodo"
] = (
    rivi_periodos["vendedores_informales"]
    / rivi_periodos["poblacion_anual"]
    * 10_000
)


indicador_rivi = (
    rivi_periodos
    .groupby(
        "codigo_localidad",
        as_index=False,
    )
    .agg(
        periodos_rivi=("archivo", "nunique"),
        vendedores_informales_promedio=(
            "vendedores_informales",
            "mean",
        ),
        rivi_por_10000_hab_2017_2019=(
            "rivi_por_10000_hab_periodo",
            "mean",
        ),
        rivi_tasa_minima=(
            "rivi_por_10000_hab_periodo",
            "min",
        ),
        rivi_tasa_maxima=(
            "rivi_por_10000_hab_periodo",
            "max",
        ),
        rivi_tasa_desviacion=(
            "rivi_por_10000_hab_periodo",
            "std",
        ),
    )
)


# =================================================
# SEGURIDAD
# =================================================
cuadrantes_audit = cuadrantes.copy()

cuadrantes_audit[
    "codigo_localidad"
] = (
    pd.to_numeric(
        cuadrantes_audit[
            "properties/PCUIULOCAL"
        ],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

indicador_cuadrantes_observado = (
    cuadrantes_audit
    .groupby(
        "codigo_localidad",
        as_index=False,
    )
    .agg(
        cuadrantes_policiales=(
            "properties/PCUNCUADRA",
            "nunique",
        )
    )
)


# Población proyectada de 2026
poblacion_2026_detalle = poblacion.loc[
    (poblacion["ANO"] == 2026)
    & (poblacion["CODIGO_LOCALIDAD"] != 0)
].copy()

poblacion_2026_detalle[
    "codigo_localidad"
] = (
    poblacion_2026_detalle[
        "CODIGO_LOCALIDAD"
    ]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

poblacion_2026 = (
    poblacion_2026_detalle
    .groupby(
        "codigo_localidad",
        as_index=False,
    )
    .agg(
        poblacion_2026=("POBLACION", "sum")
    )
)


indicador_seguridad = (
    base_territorial[
        [
            "codigo_localidad",
            "localidad",
        ]
    ]
    .merge(
        indicador_cuadrantes_observado,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
    .merge(
        poblacion_2026,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
)

indicador_seguridad[
    "cuadrantes_policiales"
] = (
    indicador_seguridad[
        "cuadrantes_policiales"
    ]
    .fillna(0)
    .astype(int)
)

indicador_seguridad[
    "cuadrantes_por_10000_hab_2026"
] = (
    indicador_seguridad[
        "cuadrantes_policiales"
    ]
    / indicador_seguridad["poblacion_2026"]
    * 10_000
)


# =================================================
# INTEGRACIÓN FINAL DE INDICADORES CRUDOS
# =================================================
columnas_finanzas_seguridad = [
    "periodos_rivi",
    "vendedores_informales_promedio",
    "rivi_por_10000_hab_2017_2019",
    "rivi_tasa_minima",
    "rivi_tasa_maxima",
    "rivi_tasa_desviacion",
    "poblacion_2026",
    "cuadrantes_policiales",
    "cuadrantes_por_10000_hab_2026",
]

base_indicadores = base_indicadores.drop(
    columns=columnas_finanzas_seguridad,
    errors="ignore",
)

base_indicadores = (
    base_indicadores
    .merge(
        indicador_rivi,
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
    .merge(
        indicador_seguridad[
            [
                "codigo_localidad",
                "poblacion_2026",
                "cuadrantes_policiales",
                "cuadrantes_por_10000_hab_2026",
            ]
        ],
        on="codigo_localidad",
        how="left",
        validate="one_to_one",
    )
)

base_indicadores = gpd.GeoDataFrame(
    base_indicadores,
    geometry="geometry",
    crs=base_territorial.crs,
)


# =================================================
# CONTROLES
# =================================================
assert len(rivi_localidades) == 120
assert (
    rivi_localidades
    .groupby("archivo")
    .size()
    .eq(20)
    .all()
)
assert (
    indicador_rivi["periodos_rivi"]
    .eq(6)
    .all()
)
assert rivi_periodos[
    "poblacion_anual"
].notna().all()

assert cuadrantes_audit[
    "properties/PCUNCUADRA"
].nunique() == 599
assert (
    base_indicadores[
        "cuadrantes_policiales"
    ].sum()
    == 599
)
assert base_indicadores[
    [
        "rivi_por_10000_hab_2017_2019",
        "cuadrantes_por_10000_hab_2026",
    ]
].notna().all().all()


resumen_finanzas_seguridad, atipicos_finanzas_seguridad = (
    auditoria_iqr(
        base_indicadores,
        [
            "rivi_por_10000_hab_2017_2019",
            "cuadrantes_por_10000_hab_2026",
        ],
    )
)


print("\nINDICADORES DE VULNERABILIDAD Y SEGURIDAD")
print("Registros RIVI locales:", len(rivi_localidades))
print(
    "Periodos RIVI por localidad:",
    sorted(
        indicador_rivi[
            "periodos_rivi"
        ].unique().tolist()
    ),
)
print(
    "Cuadrantes incluidos:",
    int(
        base_indicadores[
            "cuadrantes_policiales"
        ].sum()
    ),
)
print(
    "Localidades sin cuadrantes:",
    base_indicadores.loc[
        base_indicadores[
            "cuadrantes_policiales"
        ] == 0,
        "localidad",
    ].tolist(),
)

display(
    base_indicadores[
        [
            "codigo_localidad",
            "localidad",
            "rivi_por_10000_hab_2017_2019",
            "rivi_tasa_minima",
            "rivi_tasa_maxima",
            "cuadrantes_policiales",
            "cuadrantes_por_10000_hab_2026",
        ]
    ].round(2)
)

print("\nRESUMEN DE DISTRIBUCIONES")
display(resumen_finanzas_seguridad.round(2))

print("\nVALORES ATÍPICOS DOCUMENTADOS")
display(atipicos_finanzas_seguridad.round(2))

CONTROL DE HOMOLOGACIÓN RIVI


,archivo,filas,localidades_mapeadas,registros_no_mapeados
0,rivi-numero-vendedores-informales-localidad-20...,21,20,1
1,rivi-numero-vendedores-informales-localidad-20...,21,20,1
2,rivi-numero-vendedores-informales-localidad-20...,21,20,1
3,rivi-numero-vendedores-informales-localidad-20...,21,20,1
4,rivi-numero-vendedores-informales-localidad-20...,21,20,1
5,rivi-numero-vendedores-informales-localidad-20...,21,20,1


Nombres no mapeados:
['metropolitana o no definida']

INDICADORES DE VULNERABILIDAD Y SEGURIDAD
Registros RIVI locales: 120
Periodos RIVI por localidad: [6]
Cuadrantes incluidos: 599
Localidades sin cuadrantes: ['SUMAPAZ']


,codigo_localidad,localidad,rivi_por_10000_hab_2017_2019,rivi_tasa_minima,rivi_tasa_maxima,cuadrantes_policiales,cuadrantes_por_10000_hab_2026
0,01,USAQUEN,11.17,10.71,11.44,41,0.70
1,02,CHAPINERO,182.96,161.74,193.86,34,2.12
2,03,SANTA FE,969.43,951.95,984.27,29,2.56
3,04,SAN CRISTOBAL,92.33,91.46,92.87,31,0.79
4,05,USME,54.34,53.55,55.04,22,0.55
5,06,TUNJUELITO,62.01,61.39,62.53,13,0.74
6,07,BOSA,27.92,27.42,28.27,51,0.66
7,08,KENNEDY,44.38,42.84,46.23,75,0.68
8,09,FONTIBON,33.00,32.64,33.31,23,0.60
9,10,ENGATIVA,33.82,32.39,35.64,35,0.42



RESUMEN DE DISTRIBUCIONES


,indicador,minimo,q1,mediana,q3,maximo,limite_inferior,limite_superior,cantidad_atipicos,asimetria
0,rivi_por_10000_hab_2017_2019,11.17,33.62,59.72,136.44,1005.66,-120.62,290.68,3,2.39
1,cuadrantes_por_10000_hab_2026,0.00,0.62,0.71,1.46,7.90,-0.63,2.71,1,3.41



VALORES ATÍPICOS DOCUMENTADOS


,indicador,localidad,valor,tipo
0,rivi_por_10000_hab_2017_2019,SANTA FE,969.43,superior
1,rivi_por_10000_hab_2017_2019,LOS MARTIRES,471.24,superior
2,rivi_por_10000_hab_2017_2019,CANDELARIA,1005.66,superior
3,cuadrantes_por_10000_hab_2026,CANDELARIA,7.90,superior


### Contrato de indicadores, normalización y ensamble del IPT

Todos los indicadores se transforman a una escala de prioridad entre 0 y 1.

Para indicadores donde un valor mayor representa mayor necesidad:

\[
s_i = \frac{x_i-\min(x)}{\max(x)-\min(x)}
\]

Para indicadores de disponibilidad, donde un valor mayor representa menor
necesidad:

\[
s_i = 1-\frac{x_i-\min(x)}{\max(x)-\min(x)}
\]

La dimensión de movilidad corresponde al promedio de las puntuaciones de
estaciones y paraderos, evitando que movilidad reciba doble peso.

Ante la ausencia de ponderaciones validadas, las siete dimensiones reciben el
mismo peso:

\[
IPT_i =
100 \times \frac{1}{7}
\sum_{d=1}^{7}s_{id}
\]

Un IPT mayor representa una mayor prioridad territorial relativa dentro de las
20 localidades. El resultado no constituye una medición absoluta de pobreza,
riesgo o déficit.

In [23]:
# =================================================
# CONTRATO FINAL DE INDICADORES
# =================================================
contrato_indicadores = pd.DataFrame(
    [
        {
            "dimension": "educacion",
            "indicador":
                "oferta_regular_por_1000_pob_5_17",
            "score": "score_educacion",
            "direccion": "inversa",
            "periodo": "2025-03-31",
            "limitacion":
                "Oferta registrada, no matrícula efectiva",
        },
        {
            "dimension": "salud",
            "indicador":
                "sedes_ips_por_10000_hab",
            "score": "score_salud",
            "direccion": "inversa",
            "periodo": "Vigente 2025",
            "limitacion":
                "Sedes registradas, no hospitales ni camas",
        },
        {
            "dimension": "movilidad",
            "indicador": "estaciones_por_km2",
            "score": "score_estaciones",
            "direccion": "inversa",
            "periodo": "2025-2026",
            "limitacion":
                "Cobertura física, no tiempos de viaje",
        },
        {
            "dimension": "movilidad",
            "indicador": "paraderos_por_km2",
            "score": "score_paraderos",
            "direccion": "inversa",
            "periodo": "2025-2026",
            "limitacion":
                "Cobertura física, no frecuencia del servicio",
        },
        {
            "dimension": "ambiente",
            "indicador":
                "conflictos_ambientales_por_km2",
            "score": "score_ambiente",
            "direccion": "directa",
            "periodo": "2020-2025",
            "limitacion":
                "Registros clasificados, no severidad",
        },
        {
            "dimension": "infraestructura",
            "indicador":
                "parques_por_10000_hab_proxy",
            "score": "score_infraestructura",
            "direccion": "inversa",
            "periodo": "2024-2025",
            "limitacion":
                "Proxy de conteo; no representa área ni calidad",
        },
        {
            "dimension": "vulnerabilidad",
            "indicador":
                "rivi_por_10000_hab_2017_2019",
            "score": "score_vulnerabilidad",
            "direccion": "directa",
            "periodo": "2017-2019",
            "limitacion":
                "Indicador histórico con desfase temporal",
        },
        {
            "dimension": "seguridad",
            "indicador":
                "cuadrantes_por_10000_hab_2026",
            "score": "score_seguridad",
            "direccion": "inversa",
            "periodo": "2026-06-30",
            "limitacion":
                "Disponibilidad operativa, no incidencia delictiva",
        },
    ]
)

display(contrato_indicadores)


# =================================================
# FUNCIÓN DE NORMALIZACIÓN ORIENTADA A PRIORIDAD
# =================================================
def normalizar_prioridad_minmax(
    serie,
    direccion,
):
    valores = pd.to_numeric(
        serie,
        errors="raise",
    ).astype(float)

    if valores.isna().any():
        raise ValueError(
            f"El indicador {serie.name} contiene faltantes"
        )

    minimo = valores.min()
    maximo = valores.max()

    if maximo == minimo:
        # Sin variación territorial: puntuación neutral
        normalizado = pd.Series(
            0.5,
            index=valores.index,
            dtype=float,
        )
    else:
        normalizado = (
            (valores - minimo)
            / (maximo - minimo)
        )

    if direccion == "inversa":
        normalizado = 1 - normalizado
    elif direccion != "directa":
        raise ValueError(
            f"Dirección desconocida: {direccion}"
        )

    return normalizado


# =================================================
# NORMALIZACIÓN DE LOS OCHO INDICADORES
# =================================================
modelo_ipt = base_indicadores.copy()

for fila in contrato_indicadores.itertuples():
    modelo_ipt[fila.score] = (
        normalizar_prioridad_minmax(
            modelo_ipt[fila.indicador],
            fila.direccion,
        )
    )


# =================================================
# ENSAMBLE DE LAS SIETE DIMENSIONES
# =================================================
modelo_ipt["dim_educacion"] = (
    modelo_ipt["score_educacion"]
)

modelo_ipt["dim_salud"] = (
    modelo_ipt["score_salud"]
)

modelo_ipt["dim_movilidad"] = (
    modelo_ipt[
        [
            "score_estaciones",
            "score_paraderos",
        ]
    ].mean(axis=1)
)

modelo_ipt["dim_ambiente"] = (
    modelo_ipt["score_ambiente"]
)

modelo_ipt["dim_infraestructura"] = (
    modelo_ipt["score_infraestructura"]
)

modelo_ipt["dim_vulnerabilidad"] = (
    modelo_ipt["score_vulnerabilidad"]
)

modelo_ipt["dim_seguridad"] = (
    modelo_ipt["score_seguridad"]
)


columnas_dimensiones = [
    "dim_educacion",
    "dim_salud",
    "dim_movilidad",
    "dim_ambiente",
    "dim_infraestructura",
    "dim_vulnerabilidad",
    "dim_seguridad",
]

pesos_dimensiones = pd.Series(
    1 / len(columnas_dimensiones),
    index=columnas_dimensiones,
    name="peso",
)


# IPT base con pesos iguales
modelo_ipt["ipt_base"] = (
    modelo_ipt[columnas_dimensiones]
    .mul(pesos_dimensiones, axis=1)
    .sum(axis=1)
    * 100
)

modelo_ipt["ranking_ipt_base"] = (
    modelo_ipt["ipt_base"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)


# =================================================
# CONTROLES DE ACEPTACIÓN
# =================================================
columnas_scores = (
    contrato_indicadores["score"].tolist()
)

assert len(modelo_ipt) == 20
assert modelo_ipt["codigo_localidad"].is_unique
assert modelo_ipt[
    contrato_indicadores["indicador"].tolist()
].notna().all().all()
assert modelo_ipt[
    columnas_scores + columnas_dimensiones
].notna().all().all()
assert (
    modelo_ipt[
        columnas_scores + columnas_dimensiones
    ].ge(0).all().all()
)
assert (
    modelo_ipt[
        columnas_scores + columnas_dimensiones
    ].le(1).all().all()
)
assert modelo_ipt["ipt_base"].between(
    0,
    100,
).all()
assert abs(pesos_dimensiones.sum() - 1) < 1e-12


print("ENSAMBLE DEL IPT BASE")
print("Localidades:", len(modelo_ipt))
print("Dimensiones:", len(columnas_dimensiones))
print(
    "Peso por dimensión:",
    round(1 / len(columnas_dimensiones), 6),
)
print(
    "Suma de pesos:",
    round(pesos_dimensiones.sum(), 6),
)
print(
    "Rango IPT:",
    round(modelo_ipt["ipt_base"].min(), 2),
    "-",
    round(modelo_ipt["ipt_base"].max(), 2),
)

tabla_ipt_base = (
    modelo_ipt[
        [
            "codigo_localidad",
            "localidad",
            *columnas_dimensiones,
            "ipt_base",
            "ranking_ipt_base",
        ]
    ]
    .sort_values(
        "ranking_ipt_base"
    )
    .reset_index(drop=True)
)

display(
    tabla_ipt_base.round(3)
)

,dimension,indicador,score,direccion,periodo,limitacion
0,educacion,oferta_regular_por_1000_pob_5_17,score_educacion,inversa,2025-03-31,"Oferta registrada, no matrícula efectiva"
1,salud,sedes_ips_por_10000_hab,score_salud,inversa,Vigente 2025,"Sedes registradas, no hospitales ni camas"
2,movilidad,estaciones_por_km2,score_estaciones,inversa,2025-2026,"Cobertura física, no tiempos de viaje"
3,movilidad,paraderos_por_km2,score_paraderos,inversa,2025-2026,"Cobertura física, no frecuencia del servicio"
4,ambiente,conflictos_ambientales_por_km2,score_ambiente,directa,2020-2025,"Registros clasificados, no severidad"
5,infraestructura,parques_por_10000_hab_proxy,score_infraestructura,inversa,2024-2025,Proxy de conteo; no representa área ni calidad
6,vulnerabilidad,rivi_por_10000_hab_2017_2019,score_vulnerabilidad,directa,2017-2019,Indicador histórico con desfase temporal
7,seguridad,cuadrantes_por_10000_hab_2026,score_seguridad,inversa,2026-06-30,"Disponibilidad operativa, no incidencia delictiva"


ENSAMBLE DEL IPT BASE
Localidades: 20
Dimensiones: 7
Peso por dimensión: 0.142857
Suma de pesos: 1.0
Rango IPT: 43.71 - 70.36


,codigo_localidad,localidad,dim_educacion,dim_salud,dim_movilidad,dim_ambiente,dim_infraestructura,dim_vulnerabilidad,dim_seguridad,ipt_base,ranking_ipt_base
0,18,RAFAEL URIBE URIBE,0.813,0.967,0.391,1.000,0.801,0.021,0.932,70.357,1
1,03,SANTA FE,0.897,0.798,0.804,0.114,0.415,0.964,0.676,66.686,2
2,05,USME,0.771,1.000,0.967,0.025,0.855,0.043,0.931,65.594,3
3,11,SUBA,0.932,0.939,0.789,0.004,0.934,0.013,0.920,64.725,4
4,04,SAN CRISTOBAL,0.840,0.994,0.816,0.013,0.874,0.082,0.900,64.565,5
5,19,CIUDAD BOLIVAR,0.839,0.996,0.905,0.000,0.831,0.030,0.912,64.464,6
6,09,FONTIBON,0.907,0.905,0.751,0.019,0.893,0.022,0.924,63.150,7
7,06,TUNJUELITO,0.721,0.959,0.476,0.382,0.906,0.051,0.906,62.863,8
8,01,USAQUEN,0.962,0.712,0.736,0.054,0.929,0.000,0.911,61.497,9
9,07,BOSA,0.821,1.000,0.529,0.108,0.881,0.017,0.917,61.047,10


### Análisis de sensibilidad del IPT

Se comparan cuatro escenarios con el IPT base:

1. **Normalización por rangos:** reduce el efecto de valores extremos y conserva
   el orden relativo de las localidades.
2. **Sin proxy de parques:** excluye infraestructura porque la fuente no contiene
   área de parques.
3. **Sin RIVI:** excluye vulnerabilidad económica por su desfase temporal.
4. **Sin proxy ni RIVI:** excluye simultáneamente ambas dimensiones limitadas.

Cuando una dimensión se excluye, las dimensiones restantes vuelven a recibir
pesos iguales. El IPT base permanece como resultado principal; los escenarios
solo evalúan su estabilidad.

In [24]:
# =================================================
# NORMALIZACIÓN ROBUSTA POR RANGOS
# =================================================
def normalizar_prioridad_rangos(
    serie,
    direccion,
):
    valores = pd.to_numeric(
        serie,
        errors="raise",
    ).astype(float)

    if valores.isna().any():
        raise ValueError(
            f"El indicador {serie.name} contiene faltantes"
        )

    if valores.nunique() == 1:
        normalizado = pd.Series(
            0.5,
            index=valores.index,
            dtype=float,
        )
    else:
        rangos = valores.rank(
            method="average",
            ascending=True,
        )

        normalizado = (
            (rangos - 1)
            / (len(valores) - 1)
        )

    if direccion == "inversa":
        normalizado = 1 - normalizado
    elif direccion != "directa":
        raise ValueError(
            f"Dirección desconocida: {direccion}"
        )

    return normalizado


modelo_sensibilidad = modelo_ipt.copy()

for fila in contrato_indicadores.itertuples():
    columna_score_rango = (
        f"{fila.score}_rango"
    )

    modelo_sensibilidad[
        columna_score_rango
    ] = normalizar_prioridad_rangos(
        modelo_sensibilidad[fila.indicador],
        fila.direccion,
    )


# Dimensiones normalizadas por rangos
modelo_sensibilidad["dim_rango_educacion"] = (
    modelo_sensibilidad[
        "score_educacion_rango"
    ]
)

modelo_sensibilidad["dim_rango_salud"] = (
    modelo_sensibilidad[
        "score_salud_rango"
    ]
)

modelo_sensibilidad["dim_rango_movilidad"] = (
    modelo_sensibilidad[
        [
            "score_estaciones_rango",
            "score_paraderos_rango",
        ]
    ].mean(axis=1)
)

modelo_sensibilidad["dim_rango_ambiente"] = (
    modelo_sensibilidad[
        "score_ambiente_rango"
    ]
)

modelo_sensibilidad[
    "dim_rango_infraestructura"
] = modelo_sensibilidad[
    "score_infraestructura_rango"
]

modelo_sensibilidad[
    "dim_rango_vulnerabilidad"
] = modelo_sensibilidad[
    "score_vulnerabilidad_rango"
]

modelo_sensibilidad["dim_rango_seguridad"] = (
    modelo_sensibilidad[
        "score_seguridad_rango"
    ]
)

columnas_dimensiones_rango = [
    "dim_rango_educacion",
    "dim_rango_salud",
    "dim_rango_movilidad",
    "dim_rango_ambiente",
    "dim_rango_infraestructura",
    "dim_rango_vulnerabilidad",
    "dim_rango_seguridad",
]

modelo_sensibilidad["ipt_rangos"] = (
    modelo_sensibilidad[
        columnas_dimensiones_rango
    ].mean(axis=1)
    * 100
)

modelo_sensibilidad[
    "ranking_ipt_rangos"
] = (
    modelo_sensibilidad["ipt_rangos"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)


# =================================================
# ESCENARIOS DE EXCLUSIÓN
# =================================================
dimensiones_sin_proxy = [
    columna
    for columna in columnas_dimensiones
    if columna != "dim_infraestructura"
]

dimensiones_sin_rivi = [
    columna
    for columna in columnas_dimensiones
    if columna != "dim_vulnerabilidad"
]

dimensiones_sin_proxy_ni_rivi = [
    columna
    for columna in columnas_dimensiones
    if columna not in {
        "dim_infraestructura",
        "dim_vulnerabilidad",
    }
]


modelo_sensibilidad["ipt_sin_proxy"] = (
    modelo_sensibilidad[
        dimensiones_sin_proxy
    ].mean(axis=1)
    * 100
)

modelo_sensibilidad["ipt_sin_rivi"] = (
    modelo_sensibilidad[
        dimensiones_sin_rivi
    ].mean(axis=1)
    * 100
)

modelo_sensibilidad[
    "ipt_sin_proxy_ni_rivi"
] = (
    modelo_sensibilidad[
        dimensiones_sin_proxy_ni_rivi
    ].mean(axis=1)
    * 100
)


for columna_ipt, columna_ranking in [
    (
        "ipt_sin_proxy",
        "ranking_ipt_sin_proxy",
    ),
    (
        "ipt_sin_rivi",
        "ranking_ipt_sin_rivi",
    ),
    (
        "ipt_sin_proxy_ni_rivi",
        "ranking_ipt_sin_proxy_ni_rivi",
    ),
]:
    modelo_sensibilidad[
        columna_ranking
    ] = (
        modelo_sensibilidad[columna_ipt]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )


# =================================================
# COMPARACIÓN DE ESCENARIOS
# =================================================
escenarios = {
    "base": (
        "ipt_base",
        "ranking_ipt_base",
    ),
    "rangos": (
        "ipt_rangos",
        "ranking_ipt_rangos",
    ),
    "sin_proxy": (
        "ipt_sin_proxy",
        "ranking_ipt_sin_proxy",
    ),
    "sin_rivi": (
        "ipt_sin_rivi",
        "ranking_ipt_sin_rivi",
    ),
    "sin_proxy_ni_rivi": (
        "ipt_sin_proxy_ni_rivi",
        "ranking_ipt_sin_proxy_ni_rivi",
    ),
}

top5_base = set(
    modelo_sensibilidad.nlargest(
        5,
        "ipt_base",
    )["localidad"]
)

comparacion_escenarios = []

for nombre, (
    columna_ipt,
    columna_ranking,
) in escenarios.items():

    top5_escenario = set(
        modelo_sensibilidad.nlargest(
            5,
            columna_ipt,
        )["localidad"]
    )

    cambio_ranking = (
        modelo_sensibilidad[
            columna_ranking
        ]
        - modelo_sensibilidad[
            "ranking_ipt_base"
        ]
    ).abs()

    comparacion_escenarios.append(
        {
            "escenario": nombre,
            "dimensiones": (
                7
                if nombre in {"base", "rangos"}
                else 6
                if nombre in {
                    "sin_proxy",
                    "sin_rivi",
                }
                else 5
            ),
            "correlacion_spearman_base": (
    modelo_sensibilidad[
        "ipt_base"
    ]
    .rank(method="average")
    .corr(
        modelo_sensibilidad[
            columna_ipt
        ].rank(method="average")
    )

            ),
            "localidades_top5_compartidas":
                len(top5_base & top5_escenario),
            "cambio_medio_ranking":
                cambio_ranking.mean(),
            "cambio_maximo_ranking":
                cambio_ranking.max(),
        }
    )

comparacion_escenarios = pd.DataFrame(
    comparacion_escenarios
)


# Frecuencia de aparición en el top 5
columnas_ipt_escenarios = [
    valor[0]
    for valor in escenarios.values()
]

modelo_sensibilidad[
    "apariciones_top5"
] = 0

for columna_ipt in columnas_ipt_escenarios:
    indices_top5 = (
        modelo_sensibilidad.nlargest(
            5,
            columna_ipt,
        ).index
    )

    modelo_sensibilidad.loc[
        indices_top5,
        "apariciones_top5",
    ] += 1


columnas_ranking_escenarios = [
    valor[1]
    for valor in escenarios.values()
]

modelo_sensibilidad[
    "ranking_promedio_escenarios"
] = (
    modelo_sensibilidad[
        columnas_ranking_escenarios
    ].mean(axis=1)
)


# =================================================
# CONTROLES
# =================================================
for columna_ipt in columnas_ipt_escenarios:
    assert modelo_sensibilidad[
        columna_ipt
    ].between(0, 100).all()

assert modelo_sensibilidad[
    columnas_ranking_escenarios
].notna().all().all()


print("ESTABILIDAD GENERAL DEL IPT")
display(
    comparacion_escenarios.round(3)
)

print("\nCOMPARACIÓN POR LOCALIDAD")
tabla_sensibilidad = (
    modelo_sensibilidad[
        [
            "codigo_localidad",
            "localidad",
            "ipt_base",
            "ranking_ipt_base",
            "ipt_rangos",
            "ranking_ipt_rangos",
            "ipt_sin_proxy",
            "ranking_ipt_sin_proxy",
            "ipt_sin_rivi",
            "ranking_ipt_sin_rivi",
            "ipt_sin_proxy_ni_rivi",
            "ranking_ipt_sin_proxy_ni_rivi",
            "apariciones_top5",
            "ranking_promedio_escenarios",
        ]
    ]
    .sort_values(
        [
            "apariciones_top5",
            "ranking_promedio_escenarios",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(tabla_sensibilidad.round(2))

ESTABILIDAD GENERAL DEL IPT


,escenario,dimensiones,correlacion_spearman_base,localidades_top5_compartidas,cambio_medio_ranking,cambio_maximo_ranking
0,base,7,1.000,5,0.00,0
1,rangos,7,0.563,3,4.15,12
2,sin_proxy,6,0.838,4,2.10,12
3,sin_rivi,6,0.853,4,1.60,13
4,sin_proxy_ni_rivi,5,0.880,4,1.50,11



COMPARACIÓN POR LOCALIDAD


,codigo_localidad,localidad,ipt_base,ranking_ipt_base,ipt_rangos,ranking_ipt_rangos,ipt_sin_proxy,ranking_ipt_sin_proxy,ipt_sin_rivi,ranking_ipt_sin_rivi,ipt_sin_proxy_ni_rivi,ranking_ipt_sin_proxy_ni_rivi,apariciones_top5,ranking_promedio_escenarios
0,05,USME,65.59,3,58.65,1,62.28,3,75.80,2,73.87,2,5,2.2
1,04,SAN CRISTOBAL,64.57,5,55.64,5,60.76,5,73.97,5,71.28,5,5,5.0
2,18,RAFAEL URIBE URIBE,70.36,1,52.63,11,68.73,2,81.74,1,82.06,1,4,3.2
3,11,SUBA,64.72,4,57.14,2,59.95,7,75.30,3,71.69,4,4,4.0
4,19,CIUDAD BOLIVAR,64.46,6,50.00,13,61.36,4,74.71,4,73.04,3,3,6.0
5,03,SANTA FE,66.69,2,49.62,14,70.88,1,61.74,15,65.79,13,2,9.0
6,09,FONTIBON,63.15,7,56.77,3,58.80,9,73.31,6,70.12,6,1,6.2
7,08,KENNEDY,60.50,12,56.77,3,55.55,13,70.03,10,65.99,11,1,9.8
8,06,TUNJUELITO,62.86,8,54.89,7,58.25,10,72.49,7,68.88,7,0,7.8
9,07,BOSA,61.05,10,55.26,6,56.53,11,70.94,9,67.50,10,0,9.2


### Niveles de prioridad y robustez

El análisis de sensibilidad muestra que el IPT es estable para identificar un
grupo general de localidades prioritarias, pero no para afirmar que diferencias
pequeñas en el orden representan diferencias territoriales definitivas.

Por ello:

- `ipt_base` se conserva como puntuación continua principal.
- El nivel de prioridad se define con el ranking promedio de los cinco escenarios.
- Las 20 localidades se dividen en cuatro grupos relativos de cinco posiciones.
- La confianza indica cuántas veces aparece una localidad en el top 5.
- Las alertas son relativas al conjunto analizado y no representan emergencias
  ni umbrales absolutos de riesgo.

In [25]:
# ============================================================
# RESULTADO FINAL DE PRIORIZACIÓN
# ============================================================

modelo_final = modelo_sensibilidad.copy()

# Las métricas ya vienen incluidas en modelo_sensibilidad
modelo_final["ranking_promedio_escenarios"] = pd.to_numeric(
    modelo_final["ranking_promedio_escenarios"]
)

modelo_final["apariciones_top5"] = pd.to_numeric(
    modelo_final["apariciones_top5"]
).astype(int)


# ============================================================
# RANKING DE CONSENSO CON DESEMPATE REPRODUCIBLE
# ============================================================
# Criterios:
# 1. Mejor ranking promedio entre escenarios.
# 2. En caso de empate, mejor ranking del IPT base.
# 3. Si continúa el empate, código de localidad.

orden_consenso = (
    modelo_final
    .sort_values(
        [
            "ranking_promedio_escenarios",
            "ranking_ipt_base",
            "codigo_localidad",
        ],
        ascending=[True, True, True],
        kind="mergesort",
    )
    .index
)

ranking_unico = pd.Series(
    range(1, len(orden_consenso) + 1),
    index=orden_consenso,
    dtype="int64",
)

modelo_final["ranking_consenso"] = (
    ranking_unico
    .reindex(modelo_final.index)
    .astype(int)
)


# ============================================================
# CLASIFICACIÓN DEL NIVEL DE PRIORIDAD
# ============================================================

def clasificar_prioridad_consenso(ranking):
    """Clasifica las 20 localidades en cuatro grupos relativos."""

    if ranking <= 5:
        return "Alta"
    if ranking <= 10:
        return "Media-alta"
    if ranking <= 15:
        return "Media"
    return "Baja"


modelo_final["nivel_prioridad_consenso"] = (
    modelo_final["ranking_consenso"]
    .apply(clasificar_prioridad_consenso)
)


# ============================================================
# CONFIANZA DE LA PRIORIZACIÓN
# ============================================================

def clasificar_confianza_top5(apariciones):
    """Clasifica la estabilidad según apariciones en el top 5."""

    if apariciones >= 4:
        return "Alta"
    if apariciones >= 2:
        return "Media"
    return "Baja"


modelo_final["confianza_priorizacion"] = (
    modelo_final["apariciones_top5"]
    .apply(clasificar_confianza_top5)
)


# ============================================================
# DIMENSIONES QUE MÁS APORTAN A LA PRIORIDAD
# ============================================================

columnas_dimensiones = [
    "dim_educacion",
    "dim_salud",
    "dim_movilidad",
    "dim_ambiente",
    "dim_infraestructura",
    "dim_vulnerabilidad",
    "dim_seguridad",
]

nombres_dimensiones = {
    "dim_educacion": "Educación",
    "dim_salud": "Salud",
    "dim_movilidad": "Movilidad",
    "dim_ambiente": "Ambiente",
    "dim_infraestructura": "Infraestructura",
    "dim_vulnerabilidad": "Vulnerabilidad económica",
    "dim_seguridad": "Seguridad",
}


def obtener_dimensiones_prioritarias(fila):
    """Obtiene las dos dimensiones con mayor puntuación de prioridad."""

    orden = (
        fila[columnas_dimensiones]
        .sort_values(
            ascending=False,
            kind="mergesort",
        )
    )

    primera = orden.index[0]
    segunda = orden.index[1]

    return pd.Series(
        {
            "dimension_prioritaria_1": nombres_dimensiones[primera],
            "dimension_prioritaria_2": nombres_dimensiones[segunda],
            "score_dimension_prioritaria_1": float(orden.iloc[0]),
            "score_dimension_prioritaria_2": float(orden.iloc[1]),
        }
    )


# Permite ejecutar nuevamente la celda sin duplicar columnas
columnas_prioritarias_existentes = [
    "dimension_prioritaria_1",
    "dimension_prioritaria_2",
    "score_dimension_prioritaria_1",
    "score_dimension_prioritaria_2",
]

modelo_final = modelo_final.drop(
    columns=columnas_prioritarias_existentes,
    errors="ignore",
)

dimensiones_prioritarias = modelo_final.apply(
    obtener_dimensiones_prioritarias,
    axis=1,
)

modelo_final = pd.concat(
    [
        modelo_final,
        dimensiones_prioritarias,
    ],
    axis=1,
)


# ============================================================
# TABLA FINAL
# ============================================================

columnas_resultado = [
    "codigo_localidad",
    "localidad",
    "ipt_base",
    "ranking_ipt_base",
    "ranking_promedio_escenarios",
    "ranking_consenso",
    "apariciones_top5",
    "nivel_prioridad_consenso",
    "confianza_priorizacion",
    "dimension_prioritaria_1",
    "dimension_prioritaria_2",
    "score_dimension_prioritaria_1",
    "score_dimension_prioritaria_2",
]

resultado_priorizacion = (
    modelo_final[columnas_resultado]
    .sort_values("ranking_consenso")
    .reset_index(drop=True)
)


# ============================================================
# VALIDACIONES FINALES
# ============================================================

assert len(resultado_priorizacion) == 20
assert resultado_priorizacion["codigo_localidad"].is_unique
assert resultado_priorizacion["localidad"].is_unique
assert resultado_priorizacion["ranking_consenso"].is_unique

assert set(
    resultado_priorizacion["ranking_consenso"]
) == set(range(1, 21))

assert resultado_priorizacion[
    "nivel_prioridad_consenso"
].notna().all()

assert resultado_priorizacion[
    "confianza_priorizacion"
].notna().all()

assert resultado_priorizacion[
    "dimension_prioritaria_1"
].notna().all()

assert resultado_priorizacion[
    "dimension_prioritaria_2"
].notna().all()

assert resultado_priorizacion[
    "ipt_base"
].between(0, 100).all()


orden_niveles = [
    "Alta",
    "Media-alta",
    "Media",
    "Baja",
]

distribucion_niveles = (
    resultado_priorizacion[
        "nivel_prioridad_consenso"
    ]
    .value_counts()
    .reindex(
        orden_niveles,
        fill_value=0,
    )
    .rename("localidades")
    .to_frame()
)

distribucion_esperada = {
    "Alta": 5,
    "Media-alta": 5,
    "Media": 5,
    "Baja": 5,
}

assert (
    distribucion_niveles["localidades"].to_dict()
    == distribucion_esperada
)


# ============================================================
# PRESENTACIÓN DE RESULTADOS
# ============================================================

prioridad_alta_confianza_alta = (
    resultado_priorizacion.loc[
        (
            resultado_priorizacion[
                "nivel_prioridad_consenso"
            ].eq("Alta")
        )
        & (
            resultado_priorizacion[
                "confianza_priorizacion"
            ].eq("Alta")
        ),
        "localidad",
    ]
    .tolist()
)

print("VALIDACIÓN FINAL SUPERADA")
print()
print("RESULTADO FINAL DE PRIORIZACIÓN")
print(
    "Localidades con prioridad alta y confianza alta:",
    prioridad_alta_confianza_alta,
)

print()
print("DISTRIBUCIÓN DE NIVELES")
display(distribucion_niveles)

display(
    resultado_priorizacion.round(
        {
            "ipt_base": 2,
            "ranking_promedio_escenarios": 1,
            "score_dimension_prioritaria_1": 3,
            "score_dimension_prioritaria_2": 3,
        }
    )
)

VALIDACIÓN FINAL SUPERADA

RESULTADO FINAL DE PRIORIZACIÓN
Localidades con prioridad alta y confianza alta: ['USME', 'RAFAEL URIBE URIBE', 'SUBA', 'SAN CRISTOBAL']

DISTRIBUCIÓN DE NIVELES


,localidades
nivel_prioridad_consenso,
Alta,5
Media-alta,5
Media,5
Baja,5


,codigo_localidad,localidad,ipt_base,ranking_ipt_base,ranking_promedio_escenarios,ranking_consenso,apariciones_top5,nivel_prioridad_consenso,confianza_priorizacion,dimension_prioritaria_1,dimension_prioritaria_2,score_dimension_prioritaria_1,score_dimension_prioritaria_2
0,05,USME,65.59,3,2.2,1,5,Alta,Alta,Salud,Movilidad,1.000,0.967
1,18,RAFAEL URIBE URIBE,70.36,1,3.2,2,4,Alta,Alta,Ambiente,Salud,1.000,0.967
2,11,SUBA,64.72,4,4.0,3,4,Alta,Alta,Salud,Infraestructura,0.939,0.934
3,04,SAN CRISTOBAL,64.57,5,5.0,4,5,Alta,Alta,Salud,Seguridad,0.994,0.900
4,19,CIUDAD BOLIVAR,64.46,6,6.0,5,3,Alta,Media,Salud,Seguridad,0.996,0.912
5,09,FONTIBON,63.15,7,6.2,6,1,Media-alta,Baja,Seguridad,Educación,0.924,0.907
6,06,TUNJUELITO,62.86,8,7.8,7,0,Media-alta,Baja,Salud,Seguridad,0.959,0.906
7,03,SANTA FE,66.69,2,9.0,8,2,Media-alta,Media,Vulnerabilidad económica,Educación,0.964,0.897
8,07,BOSA,61.05,10,9.2,9,0,Media-alta,Baja,Salud,Seguridad,1.000,0.917
9,08,KENNEDY,60.50,12,9.8,10,1,Media-alta,Baja,Salud,Seguridad,0.963,0.914


## Interpretación del resultado final de priorización

El Índice de Priorización Territorial (IPT) integra siete dimensiones:
educación, salud, movilidad, ambiente, infraestructura, vulnerabilidad
económica y seguridad. Ante la ausencia de ponderaciones institucionales
validadas, cada dimensión recibió el mismo peso dentro del índice.

Los puntajes de las dimensiones representan prioridad territorial, no desempeño.
Por lo tanto, un valor normalizado más alto indica una mayor necesidad relativa,
una mayor presión territorial o una menor disponibilidad del servicio, según la
dirección definida para cada indicador.

La clasificación final combina el IPT base con cinco escenarios de sensibilidad.
Las 20 localidades se organizan en cuatro grupos relativos de cinco localidades:
prioridad alta, media-alta, media y baja. Estos niveles permiten comparar las
localidades entre sí, pero no constituyen umbrales absolutos de riesgo o emergencia.

Las localidades con prioridad alta y confianza alta son:

- Usme
- Rafael Uribe Uribe
- Suba
- San Cristóbal

Estas cuatro localidades permanecen de manera recurrente entre las cinco primeras
posiciones de los escenarios analizados. Ciudad Bolívar también pertenece al grupo
de prioridad alta, aunque presenta confianza media porque aparece en el grupo de
las cinco primeras localidades en tres de los cinco escenarios.

El orden exacto muestra sensibilidad al método de normalización. El escenario
basado en rangos presentó una correlación de Spearman de 0.563 con el IPT base,
mientras que los escenarios que excluyen el proxy de infraestructura, el indicador
histórico RIVI o ambos conservaron correlaciones entre 0.838 y 0.880. Por esta
razón, la interpretación principal debe centrarse en los niveles de prioridad,
la confianza y las dimensiones prioritarias, y no únicamente en una posición
ordinal exacta.

Cuando dos localidades presentan el mismo ranking promedio entre escenarios,
el desempate se realiza primero mediante su posición en el IPT base y después
mediante el código de localidad. Este criterio garantiza un orden único,
transparente y reproducible.

El indicador de infraestructura utiliza el número de parques por 10.000
habitantes como proxy, debido a que la fuente disponible no contiene el área,
la calidad ni la accesibilidad de los parques. El indicador RIVI corresponde
al periodo 2017–2019. Ambas limitaciones fueron evaluadas mediante escenarios
de sensibilidad.

### Alcance frente a inversión y alertas tempranas

El Plan Maestro contempla una dimensión explícita de inversión. En el catálogo de fuentes, `INV-EDU` se encuentra en estado `approved_partial`; por esta razón no se incorporó al IPT base actual mientras no se validen su alcance, unidad de medida y comparabilidad territorial para las 20 localidades. La dimensión de vulnerabilidad económica construida con RIVI no sustituye la dimensión de inversión y conserva su interpretación propia.

El resultado producido corresponde a una priorización territorial relativa. Los niveles `Alta`, `Media-alta`, `Media` y `Baja` no constituyen alertas tempranas, umbrales absolutos de riesgo ni declaraciones de emergencia. El entregable E07 — Alertas permanece pendiente de validación con datos, debido a que su implementación requiere series históricas suficientes y criterios temporales validados. Si el equipo decide incorporarlo posteriormente, deberá utilizar reglas transparentes, umbrales documentados y trazabilidad hacia los indicadores de origen.


## Exportación de resultados del IPT

Se exportan las tablas consolidadas del modelado, el contrato de indicadores,
los escenarios incorporados al modelo y el resultado final de priorización.
Los archivos constituyen la interfaz de entrega del modelado territorial para
las etapas posteriores de visualización y comunicación.

In [26]:
# ============================================================
# EXPORTACIÓN DE RESULTADOS DEL IPT
# ============================================================

from pathlib import Path


def localizar_raiz_proyecto():
    """Localiza la raíz usando las carpetas data/curated y src."""

    ubicacion_actual = Path.cwd().resolve()

    for candidato in [
        ubicacion_actual,
        *ubicacion_actual.parents,
    ]:
        if (
            (candidato / "data" / "curated").is_dir()
            and (candidato / "src").is_dir()
        ):
            return candidato

    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto."
    )


def preparar_tabla_csv(tabla):
    """Convierte la tabla y excluye la geometría del archivo CSV."""

    salida = pd.DataFrame(tabla.copy())

    if "geometry" in salida.columns:
        salida = salida.drop(columns=["geometry"])

    return salida


raiz_proyecto = localizar_raiz_proyecto()
directorio_curated = raiz_proyecto / "data" / "curated"

assert directorio_curated.is_dir()


# ============================================================
# PREPARACIÓN DE TABLAS
# ============================================================

tabla_indicadores = preparar_tabla_csv(
    base_indicadores
)

tabla_modelo_ipt = preparar_tabla_csv(
    modelo_final
)

tabla_priorizacion = preparar_tabla_csv(
    resultado_priorizacion
)

tabla_contrato = preparar_tabla_csv(
    contrato_indicadores
)


# ============================================================
# VALIDACIONES PREVIAS A LA EXPORTACIÓN
# ============================================================

assert len(tabla_indicadores) == 20
assert tabla_indicadores["codigo_localidad"].is_unique

assert len(tabla_modelo_ipt) == 20
assert tabla_modelo_ipt["codigo_localidad"].is_unique

assert len(tabla_priorizacion) == 20
assert tabla_priorizacion["codigo_localidad"].is_unique
assert tabla_priorizacion["ranking_consenso"].is_unique

assert not tabla_contrato.empty


# ============================================================
# ARCHIVOS DE SALIDA
# ============================================================

archivos_exportacion = {
    "ipt_indicadores_localidad.csv": tabla_indicadores,
    "ipt_modelo_localidad.csv": tabla_modelo_ipt,
    "ipt_priorizacion_localidades.csv": tabla_priorizacion,
    "ipt_contrato_indicadores.csv": tabla_contrato,
}


for nombre_archivo, tabla in archivos_exportacion.items():
    ruta_salida = directorio_curated / nombre_archivo

    tabla.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig",
    )

    assert ruta_salida.exists()
    assert ruta_salida.stat().st_size > 0


print("EXPORTACIÓN FINAL SUPERADA")
print()

for nombre_archivo, tabla in archivos_exportacion.items():
    ruta_relativa = (
        directorio_curated / nombre_archivo
    ).relative_to(raiz_proyecto)

    print(
        f"{ruta_relativa} | "
        f"{len(tabla)} filas | "
        f"{len(tabla.columns)} columnas"
    )

EXPORTACIÓN FINAL SUPERADA

data\curated\ipt_indicadores_localidad.csv | 20 filas | 39 columnas
data\curated\ipt_modelo_localidad.csv | 20 filas | 88 columnas
data\curated\ipt_priorizacion_localidades.csv | 20 filas | 13 columnas
data\curated\ipt_contrato_indicadores.csv | 8 filas | 6 columnas


In [27]:
# ============================================================
# VALIDACIÓN DE LOS ARCHIVOS EXPORTADOS
# ============================================================

ruta_indicadores = (
    directorio_curated / "ipt_indicadores_localidad.csv"
)

ruta_modelo = (
    directorio_curated / "ipt_modelo_localidad.csv"
)

ruta_priorizacion = (
    directorio_curated / "ipt_priorizacion_localidades.csv"
)

ruta_contrato = (
    directorio_curated / "ipt_contrato_indicadores.csv"
)


indicadores_exportados = pd.read_csv(
    ruta_indicadores,
    encoding="utf-8-sig",
    dtype={"codigo_localidad": "string"},
)

modelo_exportado = pd.read_csv(
    ruta_modelo,
    encoding="utf-8-sig",
    dtype={"codigo_localidad": "string"},
)

priorizacion_exportada = pd.read_csv(
    ruta_priorizacion,
    encoding="utf-8-sig",
    dtype={"codigo_localidad": "string"},
)

contrato_exportado = pd.read_csv(
    ruta_contrato,
    encoding="utf-8-sig",
)


# ============================================================
# CONTROLES ESTRUCTURALES
# ============================================================

codigos_esperados = {
    f"{codigo:02d}"
    for codigo in range(1, 21)
}

assert len(indicadores_exportados) == 20
assert len(modelo_exportado) == 20
assert len(priorizacion_exportada) == 20
assert len(contrato_exportado) == 8

assert indicadores_exportados[
    "codigo_localidad"
].is_unique

assert modelo_exportado[
    "codigo_localidad"
].is_unique

assert priorizacion_exportada[
    "codigo_localidad"
].is_unique

assert set(
    indicadores_exportados["codigo_localidad"]
) == codigos_esperados

assert set(
    modelo_exportado["codigo_localidad"]
) == codigos_esperados

assert set(
    priorizacion_exportada["codigo_localidad"]
) == codigos_esperados

assert priorizacion_exportada[
    "ranking_consenso"
].is_unique

assert set(
    priorizacion_exportada["ranking_consenso"]
) == set(range(1, 21))

assert priorizacion_exportada[
    "ipt_base"
].between(0, 100).all()

assert contrato_exportado[
    "dimension"
].nunique() == 7

assert not any(
    columna.startswith("Unnamed:")
    for tabla in [
        indicadores_exportados,
        modelo_exportado,
        priorizacion_exportada,
        contrato_exportado,
    ]
    for columna in tabla.columns
)


print("RELECTURA Y VALIDACIÓN DE EXPORTACIONES SUPERADA")
print()
print(
    "Indicadores:",
    indicadores_exportados.shape,
)

print(
    "Modelo IPT:",
    modelo_exportado.shape,
)

print(
    "Priorización:",
    priorizacion_exportada.shape,
)

print(
    "Contrato:",
    contrato_exportado.shape,
)

RELECTURA Y VALIDACIÓN DE EXPORTACIONES SUPERADA

Indicadores: (20, 39)
Modelo IPT: (20, 88)
Priorización: (20, 13)
Contrato: (8, 6)


## 12. Generación de Tablas Maestras por Dominio Territorial

A partir de la lógica modular en `src.modeling.domain_indicators`, se extraen y consolidan las 12 tablas maestras temáticas correspondientes a cada dominio analítico para las 20 localidades de Bogotá D.C.:

$$
\text{Tabla Maestra Dominio}_d = f_d(\text{Master Localidades})
$$

Cada tabla generada contiene la llave territorial canónica `codigo_localidad`, `nombre_localidad` y las métricas e indicadores especializados del sector.

In [ ]:
from src.modeling.domain_indicators import build_all_domain_tables

print("Generando tablas maestras temáticas por dominio...")
domain_tables = build_all_domain_tables(export_curated=True)

for domain_name, df_dom in domain_tables.items():
    print(f"✓ {domain_name:<30}: {df_dom.shape[0]} localidades x {df_dom.shape[1]} variables")

print(f"\nTotal de tablas temáticas curadas: {len(domain_tables)} en data/curated/")
